# MuZero — planifier avec un modèle appris


| | |
|---|---|
| **Idée** | Apprendre un **modèle utile pour décider** puis lancer un **MCTS** dans son espace latent. |
| **Promesse** | Planifier **sans connaître les règles exactes** de l'environnement. |
| **Démonstrations** | **CartPole-v1** pour la mécanique complète, puis **Connect-4** pour le cas adversarial. |
| **Honnêteté** | Ici on montre surtout les **mécanismes** de MuZero, pas une performance de niveau article. |

MuZero est souvent présenté comme "AlphaZero sans simulateur parfait". C'est vrai, mais trop court.
Le vrai saut conceptuel est plus précis : au lieu de reconstruire le monde comme un modèle génératif
généraliste, MuZero apprend seulement ce qui aide à **choisir une action**. Son modèle prédit
la **récompense immédiate**, la **valeur future** et une **politique prior** qui guide la recherche.

## 1. Le problème : planifier quand les règles ne sont pas accessibles

**AlphaZero** dispose d'un privilège énorme : les règles du jeu sont déjà codables comme un
simulateur exact. À partir d'une position, il peut essayer un coup, avancer d'un cran, puis
recommencer autant de fois qu'il le souhaite. **MuZero** vise des situations où ce privilège n'existe
pas, ou n'est pas pratique : Atari, contrôle, robotique, environnements dont la dynamique n'est
pas disponible comme un oracle parfait.

L'idée n'est donc pas "apprendre une image du monde". L'idée est "apprendre la partie du monde qui
sert à décider". Si un latent permet de bien prédire les récompenses, les valeurs et les priorités
d'action, il est déjà suffisant pour faire tourner un MCTS utile. C'est ce qu'on appelle souvent un
**value-equivalent model** : un modèle qui préserve ce qui compte pour l'optimisation, pas
nécessairement la fidélité visuelle.

## 2. Le cadrage honnête de ce notebook

Ce notebook suit la logique pédagogique du projet : on code une version **from scratch**, compacte,
lisible, et exécutable en quelques minutes. Cela implique quelques choix de portée.

- On reste sur des **actions discrètes**. Le cœur de MuZero suppose que le MCTS peut énumérer les
  branches. Pour un environnement continu comme HalfCheetah, il faudrait une variante du type
  **Sampled MuZero**, ce qui brouillerait le cœur du sujet.
- On construit un **petit MuZero** : fonctions `h`, `g`, `f`, cibles n-step, support catégoriel,
  bruit de Dirichlet à la racine, et démonstration CartPole.
- On n'ajoute pas ici `reanalyze`, replay priorisé, ni infrastructure massive de self-play. Ces
  extensions comptent en pratique, mais elles ne sont pas le meilleur point d'entrée.

Le bon contrat de lecture est donc : **comprendre exactement pourquoi MuZero marche** et voir un
petit signal d'apprentissage réel, sans prétendre reproduire les budgets de calcul du papier.

## 3. Plan du notebook

- **Partie I — De UCB1 à PUCT.** On reconstruit la généalogie intellectuelle du MCTS moderne :
  bandits, UCT, prior réseau, normalisation des Q.
- **Partie II — Les briques internes de MuZero.** On clarifie `h`, `g`, `f`, l'absence de
  reconstruction, le support catégoriel et l'environnement CartPole.
- **Partie III — MCTS MuZero from scratch.** Nœuds, min-max, expansion, bruit de Dirichlet,
  backup avec discount et bascule deux joueurs.
- **Partie IV — Self-play, replay et apprentissage.** On fabrique les cibles, la loss, puis un
  entraînement CartPole miniature qui montre la mécanique complète.
- **Partie V — Connect-4 adversarial.** Même moteur de recherche, mais avec masque d'actions
  légales et inversion de perspective au backup.

Gardez une boussole simple pendant toute la lecture : **le réseau propose, la recherche dispose**.

In [ ]:
import math
import random
import time
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from IPython.display import Video, display

import gymnasium as gym
from gymnasium.wrappers import RecordVideo
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from matplotlib.colors import Normalize

try:
    from pettingzoo.classic import connect_four_v3
    HAS_PETTINGZOO = True
except Exception as exc:
    HAS_PETTINGZOO = False
    PETTINGZOO_ERROR = repr(exc)



plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["font.size"] = 11

print("torch", torch.__version__)


---
# Partie I — Des bandits au MCTS

Avant de parler de MuZero, il faut comprendre la **grammaire de l'exploration**. Le MCTS moderne
n'est pas tombé du ciel : il hérite directement de la théorie des **bandits multi-bras**. Les
formules changent, mais la question reste la même : comment allouer le calcul entre ce qui semble
déjà bon et ce qui pourrait être meilleur si on regardait de plus près ?

## I.1 — UCB1 : l'idée de base

Dans un bandit à plusieurs bras, chaque bras retourne une récompense aléatoire. Si l'on exploitait
brutalement la meilleure moyenne observée, on pourrait rester bloqué sur un bras médiocre choisi
trop tôt. Si l'on explorait au hasard pour toujours, on gaspillerait une partie croissante du budget.

**Upper Confidence Bound [UCB1](https://learn.microsoft.com/en-us/archive/msdn-magazine/2019/august/test-run-the-ucb1-algorithm-for-multi-armed-bandit-problems)**
injecte un optimisme structuré :

$$
a_t = \arg\max_i \left[
\underbrace{\bar x_i}_{\text{récompense moyenne observée}}
+
\underbrace{\sqrt{\frac{2\ln t}{n_i}}}_{\text{bonus d'incertitude}}
\right].
$$

Le premier terme favorise l'**exploitation** : choisir le bras qui semble actuellement le meilleur.
Le second favorise l'**exploration** : réexaminer les bras encore mal connus. Il est grand lorsque
$n_i$, le nombre d'essais du bras $i$, est petit, puis diminue à mesure que l'on accumule des
observations. L'exploration n'est donc pas uniforme : elle est **ciblée là où une information
supplémentaire pourrait encore changer la décision**.

### Un exemple numérique

Imaginons trois machines à sous. Après quelques essais, leurs récompenses cumulées sont
$(1,\ 3,\ 0)$ et elles ont respectivement été jouées $(2,\ 4,\ 1)$ fois. Leurs moyennes empiriques
sont donc :

$$
\bar{\mathbf{x}}
=
\left(\frac{1}{2},\frac{3}{4},\frac{0}{1}\right)
=
(0{,}50,\ 0{,}75,\ 0{,}00).
$$

À l'instant $t=5$, leurs scores UCB1 valent approximativement :

$$
\begin{aligned}
\operatorname{UCB}_0 &= 0{,}50+\sqrt{\frac{2\ln 5}{2}} \approx 1{,}77,\\
\operatorname{UCB}_1 &= 0{,}75+\sqrt{\frac{2\ln 5}{4}} \approx 1{,}65,\\
\operatorname{UCB}_2 &= 0{,}00+\sqrt{\frac{2\ln 5}{1}} \approx 1{,}79.
\end{aligned}
$$

La deuxième machine possède la meilleure moyenne observée, mais UCB1 choisit ici la troisième.
Pourquoi ? Parce qu'elle n'a été essayée qu'une seule fois : son résultat nul repose sur trop peu
d'information pour conclure qu'elle est réellement mauvaise. Son fort bonus d'incertitude lui offre
une nouvelle chance. Si elle continue à décevoir, $n_2$ augmentera, son bonus diminuera et UCB1
cessera progressivement de la sélectionner.

La notion de **regret** aide à évaluer cette stratégie. Le regret cumulé mesure ce que l'on a perdu
par rapport à un agent qui aurait connu le meilleur bras dès le départ. Par exemple, si le meilleur
bras rapporte en moyenne $0{,}7$, dix tirages parfaits rapporteraient en espérance $7$. Si notre
algorithme n'obtient que $6$, son regret vaut :

$$
R_{10}=7-6=1.
$$

UCB1 accepte donc quelques choix temporairement sous-optimaux pour identifier le meilleur bras, mais
il évite d'explorer inutilement au même rythme pour toujours. Son regret croît essentiellement de
façon **logarithmique** : une fois le meilleur bras presque identifié, les autres ne sont retestés
qu'occasionnellement, juste assez pour vérifier que notre confiance reste méritée.

In [ ]:
true_means = np.array([0.15, 0.45, 0.5, 0.9, 0.35], dtype=np.float32)
K = len(true_means)
best_mean = float(true_means.max())

def pull_arm(index, rng):
    return float(rng.normal(loc=true_means[index], scale=0.45))

def run_ucb1(T=1500, seed=0):
    rng = np.random.default_rng(seed)
    counts = np.zeros(K, dtype=np.int32)
    sums = np.zeros(K, dtype=np.float64)
    regrets = []
    choices = []
    for t in range(1, T + 1):
        if (counts == 0).any():
            action = int(np.argmin(counts))
        else:
            bonus = np.sqrt(2.0 * np.log(t) / counts)
            action = int(np.argmax(sums / counts + bonus))
        reward = pull_arm(action, rng)
        counts[action] += 1
        sums[action] += reward
        regrets.append(best_mean - float(true_means[action]))
        choices.append(action)
    return np.cumsum(regrets), counts, np.asarray(choices)

def run_eps_greedy(T=1500, eps=0.1, seed=0):
    rng = np.random.default_rng(seed)
    counts = np.zeros(K, dtype=np.int32)
    sums = np.zeros(K, dtype=np.float64)
    regrets = []
    choices = []
    for _ in range(T):
        estimates = np.divide(sums, counts, out=np.zeros(K), where=counts > 0)
        if rng.random() < eps:
            action = int(rng.integers(K))
        else:
            action = int(np.argmax(estimates))
        reward = pull_arm(action, rng)
        counts[action] += 1
        sums[action] += reward
        regrets.append(best_mean - float(true_means[action]))
        choices.append(action)
    return np.cumsum(regrets), counts, np.asarray(choices)

regret_ucb, counts_ucb, choices_ucb = run_ucb1()
regret_eps, counts_eps, choices_eps = run_eps_greedy()

plt.figure(figsize=(8, 4))
plt.plot(regret_ucb, label="UCB1")
plt.plot(regret_eps, label="eps-greedy")
plt.xlabel("tour")
plt.ylabel("regret cumule")
plt.title("Bandit : regret cumule")
plt.legend()
plt.show()

**Lecture.** La courbe UCB1 finit par s'aplatir alors que `eps-greedy` continue a payer un peage
plus régulier. Ce n'est pas parce qu'UCB1 "devine mieux", mais parce qu'il **répartit son doute**
plus intelligemment. Une fois qu'un bras semble nettement dominant, il n'a plus besoin d'être
contredit très souvent pour rester crédible.

In [ ]:
labels = [f"bras {i}" for i in range(K)]
x = np.arange(K)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].bar(x, counts_ucb, color="#3b82f6")
ax[0].set_xticks(x, labels)
ax[0].set_title("Nombre de tirages UCB1")
ax[0].set_ylabel("tirages")

ax[1].bar(x, counts_eps, color="#f97316")
ax[1].set_xticks(x, labels)
ax[1].set_title("Nombre de tirages eps-greedy")
ax[1].set_ylabel("tirages")

fig.suptitle("UCB1 concentre le budget sur le meilleur bras sans ignorer les autres")
fig.tight_layout()
plt.show()
print("meilleure moyenne vraie:", int(np.argmax(true_means)))

**Lecture.** Le graphique de gauche raconte mieux la logique d'UCB1 que la seule formule. Le
meilleur bras finit par absorber l'essentiel du budget, mais les autres ne tombent pas à zéro :
ils reçoivent encore quelques tests pour vérifier que l'avantage observé n'etait pas un accident.
C'est exactement l'intuition qu'on réutilisera dans les arbres.

## I.2 — MCTS : quatre phases, même question

Dans un bandit classique, on choisit plusieurs fois entre les mêmes bras. Dans un problème
séquentiel, chaque action conduit à un **nouvel état**, dans lequel un nouveau choix apparaît. On
n'a donc plus un seul bandit, mais **un bandit dans chaque état** : chaque nœud de l'arbre doit
arbitrer entre ses propres actions.

C'est ce glissement qui donne naissance au **Monte-Carlo Tree Search**. Une simulation MCTS suit
quatre phases :

1. **Sélection** — partir de la racine et descendre dans l'arbre en choisissant, à chaque nœud,
   l'enfant qui maximise un score combinant valeur estimée et exploration.
2. **Expansion** — lorsqu'une action encore peu explorée conduit à une nouvelle situation, créer le
   nœud correspondant.
3. **Évaluation** — estimer la qualité de cette nouvelle feuille : par un rollout dans le MCTS
   classique, puis par un réseau de valeur dans AlphaZero et MuZero.
4. **Backup** — remonter cette estimation jusqu'à la racine en mettant à jour les compteurs de
   visites et les valeurs moyennes des branches traversées.

- **$N(s,a)$ — nombre de visites** : nombre de simulations ayant emprunté l’action $a$ depuis l’état $s$. Ainsi, `N=28` signifie que 28 simulations sont passées par cette branche. Le $N=40$ de la racine correspond au nombre total de simulations réalisées.

- **$Q(s,a)$ — valeur moyenne** : rendement moyen remonté par les simulations ayant emprunté cette branche. Un $Q$ élevé indique que les futurs explorés après cette action ont généralement été favorables. Ce n’est pas nécessairement un Q-network : c’est une statistique construite par les backups du MCTS.

- **$U(s,a)$ — bonus d’exploration** : bonus accordé aux actions encore insuffisamment explorées. Il est généralement élevé lorsque la branche a peu de visites, puis diminue à mesure que $N(s,a)$ augmente.

Le MCTS sélectionne la branche maximisant :

$$
\operatorname{score}(s,a)=Q(s,a)+U(s,a).
$$

Ainsi, $Q$ demande « cette branche semble-t-elle bonne ? », tandis que $U$ demande « sommes-nous suffisamment certains de notre estimation ? ». Une action peut donc être sélectionnée parce qu’elle paraît excellente, ou parce qu’elle reste trop incertaine pour être abandonnée.
```mermaid
flowchart TD
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    root["Racine<br/>état actuel<br/>$$~N=40$$"]

    left["Action gauche<br/>$$~Q=0.72,\ U=0.10,\ Q+U=0.82,\ N=28$$"]
    right["Action droite<br/>$$~Q=0.55,\ U=0.21,\ Q+U=0.76,\ N=8$$"]
    new["Action peu essayée<br/>$$~Q=0.00,\ U=0.65,\ Q+U=0.65,\ N=4$$"]

    known["Nœud déjà exploré"]
    leaf["Nouvelle feuille<br/>créée par expansion"]
    value["Évaluation de la feuille<br/>$$~v=0.68$$"]

    root -->|"1. Sélection : score maximal"| left
    root --> right
    root --> new

    left --> known
    known -->|"2. Expansion"| leaf
    leaf -->|"3. Évaluation"| value

    value -.->|"4. Backup : mise à jour de Q et N"| leaf
    leaf -.-> left
    left -.-> root

    classDef selected fill:none,stroke:#2563eb,stroke-width:3px
    classDef expanded fill:none,stroke:#16a34a,stroke-width:2px
    classDef evaluated fill:none,stroke:#d97706,stroke-width:2px

    class left selected
    class leaf expanded
    class value evaluated
```

Les nombres sont illustratifs. À la racine, l'action gauche est sélectionnée parce que son score
$Q+U$ est le plus élevé. La recherche descend ensuite dans cette branche jusqu'à rencontrer une
action qui n'a pas encore été développée. Une nouvelle feuille est créée, évaluée à $v=0{,}68$, puis
cette information remonte vers tous les ancêtres visités.

Le **backup** ne remplace pas brutalement leur ancienne estimation : il incorpore la nouvelle valeur
dans leur moyenne et incrémente leur compteur de visites. La simulation suivante repart de la même
racine, mais avec des statistiques légèrement plus informées. Le chemin sélectionné pourra donc être
différent.

L'arbre constitue ainsi une **mémoire du calcul déjà dépensé** :

- une branche prometteuse attire davantage de simulations ;
- une branche incertaine reçoit quelques essais grâce à son bonus d'exploration ;
- une branche régulièrement décevante finit par être délaissée.

Une simulation MCTS ne correspond pas à une action exécutée dans le vrai monde. L'agent peut faire
des dizaines ou des centaines de simulations internes depuis le même état, puis seulement utiliser
les statistiques obtenues à la racine pour choisir l'action réelle.

## I.3 — UCT : UCB1 appliqué localement

**UCT** (*Upper Confidence Bounds applied to Trees*) transpose UCB1 dans un arbre de recherche.
À chaque nœud, les actions disponibles sont considérées comme les bras d'un petit bandit local :

$$
a^*(s) = \arg\max_a \left[
\underbrace{Q(s,a)}_{\text{exploitation}}
+
\underbrace{c\sqrt{\frac{\ln N(s)}{N(s,a)}}}_{\text{exploration}}
\right].
$$

- $Q(s,a)$ est la valeur moyenne des retours remontés par les simulations ayant emprunté l'action
  $a$ depuis $s$. Plus cette moyenne est élevée, plus la branche semble prometteuse.
- $N(s)$ compte le nombre de simulations passées par le nœud parent. Lorsque ce nombre augmente,
  l'algorithme peut consacrer une partie de ce budget à réexaminer des branches moins visitées.
- $N(s,a)$ compte le nombre de fois où l'action $a$ a été choisie depuis $s$. Une petite valeur
  produit un grand bonus d'exploration ; ce bonus diminue progressivement lorsque la branche est
  mieux connue.
- $c$ règle le compromis entre exploitation et exploration. Un petit $c$ concentre rapidement la
  recherche sur les meilleurs $Q$ observés ; un grand $c$ répartit davantage les simulations entre
  les branches.

Une action jamais visitée pose une division par zéro. En pratique, on lui donne une priorité
infinie, ou l'on commence par essayer chaque action une fois. Cela garantit qu'aucune branche n'est
écartée avant d'avoir reçu au moins une première évaluation.

UCT applique cette règle **à chaque niveau** de l'arbre. Une simulation peut donc choisir une branche
très prometteuse à la racine, puis explorer une action encore mal connue quelques niveaux plus bas.
Chaque nœud résout son propre compromis exploration–exploitation à partir de ses statistiques
locales.

Le terme d'exploration reste cependant **sans connaissance préalable** : deux actions possédant le
même nombre de visites reçoivent le même bonus, même si l'une semble intuitivement beaucoup plus
plausible que l'autre. UCT doit découvrir cette différence uniquement par les simulations. C'est
précisément ce que PUCT modifiera en ajoutant un **prior de politique appris**.

In [ ]:
class UCTNode:
    def __init__(self):
        self.visit_count = 0
        self.value_sum = 0.0
        self.children = {}

    def value(self):
        return self.value_sum / self.visit_count if self.visit_count else 0.0

def make_leaf_values(depth, branching, seed):
    rng = np.random.default_rng(seed)
    values = {}

    def get_value(path):
        key = tuple(path)
        if key not in values:
            base = 0.2 * sum(a == 1 for a in key)
            values[key] = float(np.clip(base + rng.normal(0.0, 0.15), 0.0, 1.0))
        return values[key]

    return get_value

def uct_search(depth=4, branching=3, n_sim=300, c=1.4, seed=0):
    root = UCTNode()
    leaf_value = make_leaf_values(depth, branching, seed)
    rng = np.random.default_rng(seed)
    for _ in range(n_sim):
        node = root
        path_nodes = [node]
        path_actions = []
        current_depth = 0
        while current_depth < depth and len(node.children) == branching:
            total = max(node.visit_count, 1)
            action = max(
                range(branching),
                key=lambda a: node.children[a].value()
                + c * math.sqrt(math.log(total + 1) / (node.children[a].visit_count + 1)),
            )
            node = node.children[action]
            path_nodes.append(node)
            path_actions.append(action)
            current_depth += 1
        if current_depth < depth:
            action = len(node.children)
            node.children[action] = UCTNode()
            node = node.children[action]
            path_nodes.append(node)
            path_actions.append(action)
            current_depth += 1
        rollout = list(path_actions)
        while len(rollout) < depth:
            rollout.append(int(rng.integers(branching)))
        value = leaf_value(rollout[:depth])
        for visited in path_nodes:
            visited.visit_count += 1
            visited.value_sum += value
    return root

toy_root = uct_search()
root_visits = [toy_root.children[a].visit_count for a in sorted(toy_root.children)]
plt.figure(figsize=(5.5, 3.4))
plt.bar(range(len(root_visits)), root_visits)
plt.xticks(range(len(root_visits)), [f"action {a}" for a in range(len(root_visits))])
plt.ylabel("visites")
plt.title("UCT : concentration des visites a la racine")
plt.show()
print("visites racine:", root_visits)

**Lecture.** UCT retrouve déjà l'essentiel du comportement recherche : au lieu d'explorer toutes les
branches avec la même intensite, il dirige progressivement la plupart des simulations vers la zone
qui a produit les meilleurs retours. Mais l'évaluation de la feuille repose encore sur des
**rollouts aléatoires**, donc bruyants, parfois chers, et très dependants de l'horizon.

In [ ]:
budgets = [20, 80, 320]
fig, ax = plt.subplots(1, len(budgets), figsize=(12, 3.4), sharey=True)

for j, sims in enumerate(budgets):
    root = uct_search(n_sim=sims, seed=2)
    visits = [root.children[a].visit_count for a in sorted(root.children)]
    ax[j].bar(range(len(visits)), visits)
    ax[j].set_title(f"{sims} simulations")
    ax[j].set_xticks(range(len(visits)), [f"a{a}" for a in range(len(visits))])
    ax[j].set_xlabel("action")

ax[0].set_ylabel("visites")
fig.suptitle("UCT : la meilleure branche prend progressivement le dessus")
fig.tight_layout()
plt.show()

**Lecture.** Quand le budget grandit, la concentration ne se fait pas d'un coup magique. Au début,
l'arbre paie un coût d'orientation : il doit ouvrir des branches et recolter des indices. Puis la
branche la plus convaincante attire de plus en plus de visites. C'est la version "arbre" de la même
histoire que pour UCB1 : on dépense beaucoup au départ pour comprendre, moins ensuite pour vérifier.

## I.4 — Pourquoi les rollouts UCT sont souvent trop bruyants

UCT évalue une feuille en **terminant la trajectoire** avec une *politique de rollout* (souvent aléatoire) et prend le retour obtenu comme estimation de la valeur. C'est élégant et sans modèle… mais c'est une estimation **Monte-Carlo à un seul échantillon**, donc fragile :

- **Variance énorme.** Un rollout, c'est *un* tirage du futur. Le retour d'un rollout aléatoire oscille fortement d'un essai à l'autre ; il faut en moyenner beaucoup pour une valeur stable — or le budget de simulations est limité. Deux feuilles quasi identiques peuvent ainsi recevoir des scores très différents **par pur hasard**, et UCB, trompé, gaspille des visites.
- **L'erreur croît avec la profondeur.** Plus la feuille est loin de la fin, plus le rollout est long : la variance s'accumule et le coût explose. Dans un problème profond, simuler jusqu'au bout est à la fois lent et peu informatif.
- **Biais d'une politique de rollout faible.** Une politique aléatoire *joue mal* : la valeur estimée reflète le résultat d'un jeu médiocre, pas la vraie valeur sous jeu optimal. Dans un jeu **tactique**, où l'issue tient à quelques coups précis, le rollout rate justement ces motifs — il sur- ou sous-estime systématiquement la position.

Autrement dit, le rollout paie **deux fois** : en **variance** (trop peu d'échantillons) *et* en **biais** (politique naïve).

La parade historique : **remplacer le rollout par un réseau**. **AlphaGo** ([Silver et al., 2016](https://www.nature.com/articles/nature16961)) mélangeait déjà rollouts et **réseau de valeur** ; **AlphaZero** ([Silver et al., 2017](https://arxiv.org/abs/1712.01815)) supprime complètement les rollouts : la feuille est évaluée **en un seul passage** par un réseau qui fournit directement **une valeur** $v$ et **un prior de politique** $p$ — une estimation déterministe et apprise, ni longue ni bruitée. **MuZero** pousse l'idée d'un cran : il n'a même plus besoin du **simulateur**, puisque la dynamique elle-même est apprise (`g`).

In [ ]:
def best_visit_share(n_sim, seeds):
    shares = []
    for seed in seeds:
        root = uct_search(n_sim=n_sim, seed=seed)
        visits = np.array([root.children[a].visit_count for a in sorted(root.children)], dtype=np.float32)
        shares.append(float(visits.max() / visits.sum()))
    return np.asarray(shares)

seeds = range(20)
shares_small = best_visit_share(40, seeds)
shares_big = best_visit_share(320, seeds)

plt.figure(figsize=(7.2, 3.8))
plt.boxplot([shares_small, shares_big], tick_labels=["40 sims", "320 sims"])
plt.ylabel("part de visites de la meilleure branche")
plt.title("Avec peu de simulations, le hasard des rollouts reste dominant")
plt.show()

**Lecture.** La dispersion est instructive : avec peu de simulations, la part de visites de la
branche "gagnante" varie beaucoup d'un seed à l'autre. Le problème n'est pas seulement le manque de
budget ; c'est aussi le fait que l'évaluation terminale depend d'un **rollout stochastique** qui
ajoute une variance difficile à combattre. D'ou l'intérêt d'un estimateur appris.

## I.5 — PUCT : le réseau indique où regarder d'abord

UCT explore sans connaissance préalable : à nombre de visites égal, toutes les actions reçoivent le
même bonus, même si certaines paraissent beaucoup plus plausibles que d'autres. Cette stratégie
fonctionne, mais elle peut gaspiller de nombreuses simulations avant de découvrir les branches
intéressantes.

**PUCT** ajoute une information supplémentaire : le **prior de politique** $P(s,a)$ prédit par le
réseau. Ce prior représente la plausibilité initiale de chaque action avant que la recherche ait
dépensé beaucoup de calcul dans ce nœud.

Dans sa forme intuitive, le bonus devient :

$$
U(s,a)
=
c_{\text{puct}}\,
\underbrace{P(s,a)}_{\text{préférence initiale du réseau}}
\frac{\sqrt{\sum_b N(s,b)}}{1+N(s,a)}.
$$

La sélection maximise alors :

$$
a^*(s)=\arg\max_a\left[Q_{\mathrm{norm}}(s,a)+U(s,a)\right].
$$

Les deux termes répondent à des questions différentes :

- $Q_{\mathrm{norm}}(s,a)$ demande : **« les simulations déjà réalisées dans cette branche ont-elles
  produit de bons résultats ? »**
- $U(s,a)$ demande : **« cette action mérite-t-elle encore du calcul compte tenu du prior et du
  nombre de visites ? »**

À $Q$ et nombres de visites égaux, une action de prior $P=0{,}7$ sera donc examinée avant une action
de prior $P=0{,}1$. Mais cette préférence n'est pas définitive : chaque visite augmente $N(s,a)$,
ce qui réduit progressivement le bonus de l'action. Si ses simulations sont décevantes, son $Q$
restera faible et une autre branche pourra prendre le dessus.

Le prior agit ainsi comme une **carte approximative** : il indique où commencer les recherches, mais
les résultats découverts dans l'arbre restent capables de le contredire.

### La formule exacte utilisée par MuZero

MuZero remplace la constante fixe $c_{\text{puct}}$ par un coefficient qui évolue lentement avec le
nombre total de visites du parent :

$$
U(s,a)
=
P(s,a)\,
\frac{\sqrt{\sum_b N(s,b)}}{1+N(s,a)}
\left(
c_1+
\log\frac{\sum_bN(s,b)+c_2+1}{c_2}
\right),
\qquad
c_1=1{,}25,\quad c_2=19652.
$$

| Terme | Rôle |
|---|---|
| $P(s,a)$ | prior fourni par la tête de politique ; oriente les premières simulations |
| $\sum_bN(s,b)$ | budget total déjà dépensé dans le nœud parent |
| $\sqrt{\sum_bN(s,b)}$ | fait remonter progressivement la pression d'exploration lorsque le parent reçoit davantage de calcul |
| $1+N(s,a)$ | réduit le bonus d'une action à mesure qu'elle est visitée ; le `+1` évite une division par zéro |
| $c_1$ | fixe l'influence initiale du prior |
| $c_2$ | contrôle la vitesse de croissance de la correction logarithmique |

Comme $c_2$ est très grand, le logarithme évolue lentement aux petits budgets. Le coefficient reste
donc proche de $c_1$ au début, puis renforce légèrement l'exploration dans les arbres qui reçoivent
beaucoup de simulations.

PUCT crée finalement une boucle d'amélioration :

```text
le réseau propose un prior P
        ↓
le MCTS répartit les simulations avec Q + U
        ↓
les visites produisent une politique améliorée π
        ↓
le réseau apprend à rapprocher P de π
```

Le réseau aide donc la recherche à devenir plus efficace, tandis que la recherche fournit au réseau
une cible meilleure que sa propre politique initiale. C'est cette coopération qui résume la formule :
**le réseau propose, la recherche dispose**.

In [ ]:
def puct_bonus(prior, child_N, parent_N, c1=1.25, c2=19652):
    pb_c = math.log((parent_N + c2 + 1.0) / c2) + c1
    return prior * pb_c * (math.sqrt(max(parent_N, 1.0)) / (child_N + 1.0))

priors = np.array([0.55, 0.25, 0.15, 0.05], dtype=np.float32)
child_visits = np.arange(0, 30)
parent_N = 40

plt.figure(figsize=(8.2, 4.0))
for i, prior in enumerate(priors):
    bonuses = [puct_bonus(float(prior), int(n), parent_N) for n in child_visits]
    plt.plot(child_visits, bonuses, label=f"P={prior:.2f}")
plt.xlabel("N(s,a)")
plt.ylabel("bonus U")
plt.title("PUCT : prior eleve au debut, puis depreciation par les visites")
plt.legend()
plt.show()

**Lecture.** Cette figure est la bonne façon de lire PUCT. A `Q` egal, l'action de prior eleve est
testee en premier. Ensuite, le bonus s'effondre en `1 / (1 + N)`, ce qui permet aux autres actions de
recuperer une chance d'être examinees. Le prior agit donc comme un **biais d'amorcage**, pas comme un
verdict irreformable.

## I.6 — Pourquoi MuZero normalise les Q

Dans un arbre réel, les valeurs `Q` peuvent vivre sur des échelles très différentes selon les
branches, surtout au début. Si l'on additionne directement `Q + U`, le bonus d'exploration peut être
soit écrasé, soit disproportionné. MuZero utilise donc une **normalisation min-max** sur les Q
observés dans l'arbre :

$$
Q_{norm} = \frac{Q - Q_{min}}{Q_{max} - Q_{min}}.
$$

Ce n'est pas un détail cosmétique : cela rend le terme de valeur et le terme de prior comparables
pendant la recherche.

In [ ]:
raw_q = np.array([-2.0, 0.75, 5.0], dtype=np.float32)
q_min, q_max = float(raw_q.min()), float(raw_q.max())
normalized_q = (raw_q - q_min) / (q_max - q_min)

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.4))
ax[0].bar(range(len(raw_q)), raw_q, color="#64748b")
ax[0].set_xticks(range(len(raw_q)), [f"a{i}" for i in range(len(raw_q))])
ax[0].set_title("Q bruts")

ax[1].bar(range(len(normalized_q)), normalized_q, color="#10b981")
ax[1].set_xticks(range(len(normalized_q)), [f"a{i}" for i in range(len(normalized_q))])
ax[1].set_title("Q normalises dans [0,1]")

fig.suptitle("La normalisation ne change pas l'ordre, elle change l'echelle de comparaison")
fig.tight_layout()
plt.show()

**Lecture.** La normalisation min-max ne "corrige" pas la valeur ; elle l'exprime dans une monnaie
commune pour la sélection. On garde le classement relatif des actions, mais on évite qu'une échelle
arbitraire vienne etouffer le bonus de prior ou, inversement, qu'un bonus trop grand ignore les
differences de valeur.

In [ ]:
root = uct_search(n_sim=220, seed=4)
child_ids = sorted(root.children)

fig, ax = plt.subplots(figsize=(9.0, 5.5))
ax.scatter([0], [0], s=1600, color="#1d4ed8")
ax.text(0, 0, f"root\nN={root.visit_count}", ha="center", va="center", color="white", fontsize=11)

xs = np.linspace(-2.0, 2.0, len(child_ids))
for x, action in zip(xs, child_ids):
    child = root.children[action]
    ax.plot([0, x], [0, -1], color="#64748b")
    ax.scatter([x], [-1], s=1050, color="#0ea5e9")
    ax.text(
        x,
        -1,
        f"a{action}\nN={child.visit_count}\nQ={child.value():.2f}",
        ha="center",
        va="center",
        color="white",
        fontsize=9,
    )
    grand_ids = sorted(child.children)[:2]
    gx = np.linspace(x - 0.45, x + 0.45, max(1, len(grand_ids)))
    for gx_i, g_action in zip(gx, grand_ids):
        grand = child.children[g_action]
        ax.plot([x, gx_i], [-1, -2], color="#94a3b8", alpha=0.8)
        ax.scatter([gx_i], [-2], s=650, color="#22c55e")
        ax.text(gx_i, -2, f"{g_action}\nN={grand.visit_count}", ha="center", va="center", fontsize=8)

ax.set_title("Un petit arbre UCT : les visites rendent visible l'attention du calcul")
ax.axis("off")
plt.show()

**Lecture.** Ce schema rappelle un point souvent cache par les équations : un arbre MCTS est un
**budget de pensee matérialise**. Les nombres de visites ne sont pas des probabilités magiques ; ils
disent simplement où l'algorithme a choisi de dépenser son temps de calcul. C'est pourquoi MuZero
reutilise ensuite ces visites comme cible de politique.

A ce stade, la transition vers MuZero devient naturelle. UCB1 explique comment explorer avec
parcimonie. UCT transporte cette idée dans un arbre. PUCT ajoute un **prior réseau** pour guider la
recherche des ses premiers pas. Il nous manque encore trois ingredients : un **état latent**, une
**dynamique apprise**, et une **valeur apprise** qui remplacent le simulateur parfait.

**La généalogie en un schéma :**

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    ucb1["UCB1<br/>bandit : $$~Q + c\sqrt{\tfrac{\ln N}{n}}$$"]
    uct["UCT<br/>UCB1 à chaque nœud de l'arbre"]
    puct["PUCT<br/>prior réseau + $$~Q$$ normalisé"]
    mz["MuZero<br/>PUCT dans un modèle appris $$~(h,g,f)$$"]
    ucb1 --> uct --> puct --> mz
```

---
# Partie II — Les briques internes de MuZero

MuZero se comprend beaucoup mieux si l'on refuse les slogans et que l'on se force a nommer les trois
fonctions qu'il apprend. Tout le reste du notebook n'est qu'une conséquence de cette décomposition.

## II.1 — Les trois fonctions `h`, `g`, `f`

Le cœur de MuZero est un **modèle de monde latent** composé de trois fonctions apprises
conjointement :

$$
\underbrace{s_t^0=h(o_t)}_{\text{représentation}},
\qquad
\underbrace{(s_t^{k+1},\hat r_t^{k+1})=g(s_t^k,a_t^k)}_{\text{dynamique}},
\qquad
\underbrace{(p_t^k,v_t^k)=f(s_t^k)}_{\text{prédiction}}.
$$

Ces fonctions ne sont pas trois modèles indépendants assemblés après coup. Elles forment une seule
chaîne différentiable, entraînée de bout en bout pour produire les quantités dont le MCTS a besoin :
des récompenses, des valeurs et des priorités d'action.

### `h` — représenter l'état réel

La fonction de représentation reçoit l'observation réelle et construit le latent racine :

$$
s_t^0=h(o_t).
$$

Dans notre CartPole, $o_t$ contient la position et la vitesse du chariot, ainsi que l'angle et la
vitesse angulaire du poteau. `h` transforme ces quatre nombres en un vecteur latent plus grand, dont
les dimensions n'ont pas nécessairement une interprétation physique directe.

Ce premier latent est particulier : c'est le seul état de l'arbre directement ancré dans une
observation réelle. À chaque décision prise dans l'environnement, MuZero appelle `h` une fois pour
créer la racine du nouveau MCTS.

Dans le papier original, `h` peut recevoir un historique d'observations et d'actions lorsque
l'observation courante ne suffit pas à décrire l'état. CartPole étant presque markovien, une seule
observation suffit ici.

### `g` — imaginer les conséquences d'une action

La fonction de dynamique reçoit un latent et une action hypothétique :

$$
(s_t^{k+1},\hat r_t^{k+1})=g(s_t^k,a_t^k).
$$

Elle produit deux sorties :

- un **nouvel état latent** $s_t^{k+1}$, représentant la situation utile après l'action ;
- une **récompense prédite** $\hat r_t^{k+1}$ pour cette transition.

Le point essentiel est que `g` ne reçoit pas la prochaine observation réelle. Pendant le MCTS, cette
observation n'existe pas encore : l'action n'a pas été exécutée dans l'environnement. Le modèle doit
donc avancer uniquement à partir de son latent précédent et de l'action envisagée.

Sur CartPole, si l'arbre envisage « pousser à droite », `g` ne cherche pas nécessairement à prédire
les quatre coordonnées physiques exactes du prochain état. Il doit produire un latent permettant de
répondre correctement à des questions utiles :

- cette action rapproche-t-elle le poteau de l'équilibre ?
- augmente-t-elle le risque de sortie du chariot ?
- combien de récompense cette transition devrait-elle rapporter ?
- quelles actions deviendront prometteuses ensuite ?

Dans MuZero, la dynamique est déterministe : un même couple $(s,a)$ produit toujours le même latent
suivant. L'incertitude n'est donc pas représentée explicitement comme dans PETS ou Dreamer.

### `f` — guider et évaluer la recherche

La fonction de prédiction lit n'importe quel latent de l'arbre :

$$
(p_t^k,v_t^k)=f(s_t^k).
$$

Elle produit :

- $p_t^k$, un **prior de politique** sur les actions ;
- $v_t^k$, une **estimation de la valeur** de cet état latent.

Le prior $p_t^k$ indique au PUCT quelles actions examiner en premier. Il ne choisit pas directement
l'action finale : le MCTS peut le corriger en découvrant des branches dont les valeurs sont meilleures
ou moins bonnes que prévu.

La valeur $v_t^k$ sert à évaluer une nouvelle feuille. Au lieu de poursuivre la simulation jusqu'à
la fin de l'épisode avec des actions aléatoires, MuZero demande au réseau : « à partir de ce latent,
quel rendement futur peut-on espérer ? » Cette estimation est ensuite remontée vers la racine par le
backup.

### Initial inference et recurrent inference

Les trois fonctions interviennent selon deux modes :

- **Initial inference** : `h` encode l'observation réelle, puis `f` évalue la racine.
- **Recurrent inference** : `g` avance d'un pas latent à partir d'une action hypothétique, puis `f`
  évalue le nouvel état.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    subgraph initial["Initial inference — ancrage dans le réel"]
        obs["observation réelle<br/>$$~o_t$$"]
        repr["représentation<br/>$$~h$$"]
        root["latent racine<br/>$$~s_t^0$$"]
        pred0["prédiction<br/>$$~f$$"]
        out0["prior et valeur<br/>$$~p_t^0,\ v_t^0$$"]

        obs --> repr --> root --> pred0 --> out0
    end

    subgraph recurrent["Recurrent inference — imagination dans l'arbre"]
        action["action hypothétique<br/>$$~a_t^0$$"]
        dynamics["dynamique apprise<br/>$$~g$$"]
        next["latent suivant<br/>$$~s_t^1$$"]
        reward["récompense prédite<br/>$$~\hat r_t^1$$"]
        pred1["prédiction<br/>$$~f$$"]
        out1["prior et valeur<br/>$$~p_t^1,\ v_t^1$$"]

        action --> dynamics
        dynamics --> next
        dynamics --> reward
        next --> pred1 --> out1
    end

    root --> dynamics
    next -.->|"nouvelle action hypothétique"| dynamics
```

La boucle de droite peut être répétée pour construire une branche entière :

```text
s⁰ --a⁰--> s¹ --a¹--> s² --a²--> s³ ...
```

Une simulation MCTS appelle donc `h` uniquement à la racine, puis alterne `g` et `f` dans l'espace
latent. Aucune observation future ni aucun simulateur externe n'est nécessaire.

| Étape | Fonction | Information disponible | Sortie utile au MCTS |
|---|---|---|---|
| Racine réelle | `h(o)` | observation réelle | latent initial |
| Évaluation d'un nœud | `f(s)` | latent courant | prior $p$, valeur $v$ |
| Transition imaginée | `g(s,a)` | latent + action | latent suivant, récompense |
| Nouvelle feuille | `f(s')` | nouveau latent | prior et valeur de feuille |

---



## II.2 — Pourquoi MuZero ne reconstruit pas l'observation

Cette architecture ressemble à un modèle de monde, mais elle ne cherche pas à reproduire fidèlement
tout ce qui existe dans le monde.

Dreamer apprend notamment un décodeur capable de reconstruire l'observation :

$$
s_t \longrightarrow \hat o_t.
$$

MuZero ne possède pas cette contrainte. Aucun décodeur ne lui demande de transformer $s_t^k$ en image,
en plateau ou en coordonnées physiques. Son latent est uniquement entraîné à prédire :

$$
\text{récompense},\qquad \text{politique de recherche},\qquad \text{valeur}.
$$

Le modèle n'est donc pas sanctionné parce qu'il oublie un détail visuel ou physique. Il est sanctionné
si cet oubli empêche de prédire correctement :

1. ce que rapporte une action ;
2. quelles actions méritent d'être explorées ;
3. si le futur de cette branche est favorable.

On parle de **suffisance décisionnelle** ou de modèle *value-equivalent*. Deux états physiques peuvent
avoir des représentations latentes proches si leurs conséquences pour la décision sont similaires.
Inversement, une petite différence visuelle doit être conservée si elle change radicalement la bonne
action.



## II.3 — CartPole comme premier laboratoire
### Exemple : ce que le latent doit retenir sur CartPole

Avant d'entraîner quoi que ce soit, regardons explicitement l'environnement. CartPole est un bon
terrain pédagogique pour MuZero car tout y est discret du côté action, mais non trivial du côté
dynamique.

- Observation : `position du chariot`, `vitesse`, `angle du pole`, `vitesse angulaire`.
- Actions : `0 = pousser a gauche`, `1 = pousser a droite`.
- Récompense : `+1` par pas tant que l'épisode survit.
- Fin : quand le pole tombé, que le chariot sort de la zone autorisee, ou que l'horizon est atteint.

Cette structure permet de visualiser très concrètement ce que doit apprendre le modèle latent.

Imaginons deux observations presque identiques :

- dans la première, le poteau penche légèrement à droite mais revient vers le centre ;
- dans la seconde, il penche au même angle mais continue à tomber rapidement.

Visuellement, leurs angles sont proches. Pourtant, leurs vitesses angulaires imposent probablement des
actions différentes. Le latent doit donc retenir cette différence.

À l'inverse, une variation minuscule de la position du chariot peut être ignorée si elle ne modifie ni
la récompense attendue, ni la valeur, ni la meilleure action. MuZero choisit ainsi ce qu'il mémorise
en fonction de son utilité pour la planification.

### Ce que cela apporte

Ne pas reconstruire l'observation permet de consacrer la capacité du réseau aux informations utiles
pour décider. Sur des images complexes, il n'est pas nécessaire de modéliser précisément chaque
texture, couleur ou détail d'arrière-plan si ces éléments ne changent pas les actions optimales.

Mais ce choix a aussi une contrepartie : on ne peut pas regarder le latent et exiger qu'il corresponde
à un état physique compréhensible. Le modèle peut même apprendre une dynamique très différente de la
véritable dynamique, tant que la recherche effectuée dans cette dynamique produit les bonnes
récompenses, valeurs et décisions.

MuZero n'apprend donc pas nécessairement **le monde tel qu'il est**. Il apprend un monde interne dans
lequel il est possible de **planifier correctement**. C'est cette liberté qui constitue à la fois sa
force principale et la difficulté conceptuelle centrale de la méthode.



In [ ]:
cartpole_demo = gym.make("CartPole-v1")
obs, info = cartpole_demo.reset(seed=0)
print("observation initiale:", np.round(obs, 3))
print("dimension observation:", cartpole_demo.observation_space.shape[0])
print("nombre d'actions:", cartpole_demo.action_space.n)
print("actions:", {0: "gauche", 1: "droite"})
cartpole_demo.close()

**Lecture.** MuZero ne voit pas "un pole" au sens symbolique. Il reçoit un vecteur continu de quatre
nombres. `h` doit en extraire un latent qui retient ce qui sert à prédire : le pole est-il en train de
se redresser ou de tomber ? Quelle action changera le signe de l'acceleration utile ? Toute la suite
du notebook peut se lire comme une maniere d'organiser proprement cette prédiction.

In [ ]:
env = gym.make("CartPole-v1")
obs, _ = env.reset(seed=3)
positions, angles, rewards = [], [], []

for _ in range(25):
    positions.append(float(obs[0]))
    angles.append(float(obs[2]))
    action = env.action_space.sample()
    obs, reward, terminated, truncated, _ = env.step(action)
    rewards.append(float(reward))
    if terminated or truncated:
        break

env.close()

fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(positions, marker="o")
ax[0].set_title("position du chariot")
ax[0].set_xlabel("temps")
ax[0].set_ylabel("x")
ax[1].plot(angles, marker="o", color="#ef4444")
ax[1].set_title("angle du pole")
ax[1].set_xlabel("temps")
ax[1].set_ylabel("theta")
fig.suptitle("Episode aleatoire court : ce que l'agent observe reellement")
fig.tight_layout()
plt.show()
print("longueur de l'episode inspecte:", len(rewards), "| retour:", sum(rewards))

**Lecture.** En aléatoire, les courbes dégénèrent vite. C'est exactement ce que le modèle doit
apprendre : relier une petite variation de vitesse ou d'angle a un futur plus ou moins durable. Sur
CartPole, la récompense est simple mais l'information predictive n'est pas "visuellement evidente" ;
elle vit dans l'interaction des quatre dimensions.

## II.4 — La transformation scalaire : compresser les grandes valeurs

MuZero ne régresse pas directement les scalaires de récompense et de valeur. Il applique souvent une
transformation non linéaire :

$$
h(x) = \operatorname{sign}(x)(\sqrt{|x| + 1} - 1) + \varepsilon x,
$$

puis la transformation inverse au décodage. Pourquoi ? Parce que les cibles de valeur peuvent avoir
des échelles très différentes : une erreur de `100` ne doit pas écraser toutes les petites erreurs
autour de zéro. Cette compression conserve le signe et l'ordre des valeurs, tout en rapprochant les
grandes amplitudes. Comme elle est inversible, le réseau peut ensuite restituer une récompense ou une
valeur dans son échelle originale. En pratique, MuZero utilise généralement
$\varepsilon = 0{,}001$ pour conserver une légère composante linéaire.

In [ ]:
# Mini-test : transform scalaire et inverse
def scalar_transform(x, eps=1e-3):
    return torch.sign(x) * (torch.sqrt(torch.abs(x) + 1.0) - 1.0) + eps * x

def inverse_scalar_transform(y, eps=1e-3):
    ay = torch.abs(y)
    inside = torch.sqrt(1.0 + 4.0 * eps * (ay + 1.0 + eps))
    base = ((inside - 1.0) / (2.0 * eps)) ** 2 - 1.0
    return torch.sign(y) * base

grid = torch.linspace(-25, 25, 300)
transformed = scalar_transform(grid)
recovered = inverse_scalar_transform(transformed)

assert torch.allclose(recovered, grid, atol=2e-3)
plt.figure(figsize=(8, 3.8))
plt.plot(grid.numpy(), transformed.numpy(), label="transform")
plt.plot(grid.numpy(), recovered.numpy(), label="inverse(transform(x))", alpha=0.8)
plt.legend()
plt.title("Compression scalaire puis reconstruction")
plt.xlabel("x")
plt.ylabel("valeur")
plt.show()
print("round-trip max error:", float((recovered - grid).abs().max()))

**Lecture.** La courbe de transform est quasi linéaire près de zéro, puis se tasse pour les grandes
amplitudes. C'est exactement ce qu'on veut : garder de la sensibilite là où les petites differences
comptent, sans laisser quelques grosses cibles dominer numeriquement toute l'optimisation.

## II.5 — Support catégoriel et two-hot sur CartPole

MuZero utilise le support catégoriel uniquement pour prédire les **récompenses** et les **valeurs**.
La politique possède, elle, deux sorties classiques correspondant aux actions gauche et droite.

Sur CartPole, la récompense immédiate vaut presque toujours `+1`, tandis que la valeur d'un état
représente le nombre actualisé de pas pendant lesquels l'agent pense pouvoir encore maintenir le
poteau. Cette valeur peut donc varier de quelques pas jusqu'à environ `500`.

Au lieu de prédire directement cette valeur réelle, le réseau produit une distribution sur un
support entier :

$$
\mathcal S=\{-S,-S+1,\ldots,0,\ldots,S-1,S\}.
$$

Dans notre entraînement, nous choisissons `support_size = 25`, soit $51$ classes entre `-25` et
`+25`. Ces bornes sont appliquées **après** la transformation scalaire.

### Exemple : un état qui semble pouvoir survivre encore 100 pas

Supposons que la cible de valeur n-step soit :

$$
x=100.
$$

On commence par la compresser avec la transformation utilisée par MuZero, avec
$\varepsilon=0{,}001$ :

$$
h(100)
=
\sqrt{101}-1+0{,}001\times100
\approx 9{,}15.
$$

La cible transformée `9.15` se situe entre les classes `9` et `10`. Son encodage two-hot place donc :

- une masse `0.85` sur la classe `9` ;
- une masse `0.15` sur la classe `10` ;
- une masse nulle sur les autres classes.

| Classe du support | $\cdots$ | $8$ | $9$ | $10$ | $11$ | $\cdots$ |
|---|---:|---:|---:|---:|---:|---:|
| Cible two-hot | $\cdots$ | 0 | **0.85** | **0.15** | 0 | $\cdots$ |

Cette interpolation conserve la valeur transformée :

$$
0{,}85\times9+0{,}15\times10=9{,}15.
$$

Le réseau apprend cette distribution avec une cross-entropy. Lors du décodage, on calcule
l'espérance du support, puis on applique la transformation inverse :

$$
9{,}15
\overset{h^{-1}}{\longmapsto}
100.
$$

Le réseau peut donc représenter une valeur continue, même si ses sorties correspondent à des classes
entières.

### Exemple : la récompense immédiate `+1`

La tête de récompense utilise exactement le même mécanisme. Pour une transition normale de CartPole :

$$
r=1
\qquad\Longrightarrow\qquad
h(1)=\sqrt{2}-1+0{,}001\approx0{,}415.
$$

La cible two-hot place alors environ :

- `0.585` sur la classe `0` ;
- `0.415` sur la classe `1`.

| Classe du support | $-1$ | $0$ | $1$ | $2$ |
|---|---:|---:|---:|---:|
| Cible de récompense | 0 | **0.585** | **0.415** | 0 |

Cela ne signifie pas que la récompense vaut parfois `0` et parfois `1`. Les deux masses constituent
une **interpolation numérique** dont l'espérance redonne la récompense transformée exacte.

### Pourquoi choisir un support de 25 ?

Le retour maximal de CartPole vaut `500`. Après transformation :

$$
h(500)
=
\sqrt{501}-1+0{,}001\times500
\approx21{,}88.
$$

Cette valeur tient dans le support `[-25, 25]`. Un support limité à `[-10,10]` saturerait au contraire
bien avant la valeur maximale : plusieurs états capables de survivre longtemps deviendraient
indiscernables pour la tête de valeur.

La chaîne complète pour notre environnement est donc :

```text
valeur CartPole x = 100
        ↓ transformation scalaire
h(x) ≈ 9.15
        ↓ encodage two-hot
0.85 sur 9, 0.15 sur 10
        ↓ cross-entropy
distribution prédite par le réseau
        ↓ espérance du support
ŷ ≈ 9.15
        ↓ transformation inverse
x̂ ≈ 100 pas de survie
```

Le two-hot transforme ainsi une régression de grande amplitude en une classification douce, tout en
conservant une estimation continue et décodable dans l'unité naturelle de CartPole : le nombre de
pas de survie attendu.

In [ ]:
# Mini-test : support categoriel two-hot
def scalar_to_support(x, support_size, apply_transform=True):
    values = scalar_transform(x) if apply_transform else x
    values = values.clamp(-support_size, support_size)
    lower = values.floor()
    upper = values.ceil()
    prob_upper = values - lower
    prob_lower = 1.0 - prob_upper
    bins = torch.zeros(*values.shape, 2 * support_size + 1, dtype=values.dtype, device=values.device)
    lower_idx = (lower + support_size).long()
    upper_idx = (upper + support_size).long()
    bins.scatter_add_(-1, lower_idx.unsqueeze(-1), prob_lower.unsqueeze(-1))
    bins.scatter_add_(-1, upper_idx.unsqueeze(-1), prob_upper.unsqueeze(-1))
    return bins

def support_to_scalar(logits, support_size, apply_inverse=True):
    probs = torch.softmax(logits, dim=-1)
    support = torch.arange(-support_size, support_size + 1, dtype=probs.dtype, device=probs.device)
    values = (probs * support).sum(dim=-1)
    return inverse_scalar_transform(values) if apply_inverse else values

support_size = 10
vals = torch.tensor([-8.0, -1.2, 0.0, 3.7, 9.1])
encoded = scalar_to_support(vals, support_size)
decoded = support_to_scalar(torch.log(encoded + 1e-9), support_size)

assert torch.allclose(decoded, vals, atol=0.35)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.3))
ax[0].bar(np.arange(encoded.shape[-1]) - support_size, encoded[3].numpy())
ax[0].set_title("Two-hot pour x = 3.7")
ax[1].bar(np.arange(encoded.shape[-1]) - support_size, encoded[1].numpy(), color="#f97316")
ax[1].set_title("Two-hot pour x = -1.2")
fig.tight_layout()
plt.show()
print("decoded:", np.round(decoded.numpy(), 2))

**Lecture.** Les deux barres non nulles valent la peine d'être contemplees : elles disent que la
cible ne "choisit pas" brutalement un entier voisin, elle partage sa masse de façon linéaire. On
obtient ainsi une tête de valeur et de récompense qui apprend plus facilement que si on l'obligeait a
tomber exactement sur une classe unique.

## II.6 — Les réseaux `h`, `g`, `f` en version compacte

Pour rester lisibles et adaptés aux observations vectorielles de CartPole, nous utilisons de petits
MLP. Sur des images Atari ou des plateaux complexes, les mêmes fonctions seraient généralement
implémentées avec des réseaux convolutionnels et des blocs résiduels ; leur rôle algorithmique
resterait toutefois identique.

- `Representation` encode l'observation réelle $o$ en un latent racine $s^0$.
- `Dynamics` reçoit $(s^k,a^k)$, puis retourne le latent suivant $s^{k+1}$ et les logits de la
  récompense prédite.
- `Prediction` lit n'importe quel latent et retourne deux têtes : les logits de politique sur les
  deux actions et les logits de valeur sur le support catégoriel.

L'action est injectée dans `Dynamics` en **one-hot**. On représente donc gauche par `[1, 0]` et droite
par `[0, 1]`, plutôt que par les nombres `0` et `1`. Cela évite de suggérer au réseau une fausse
relation numérique entre les actions : ce sont deux choix distincts, pas deux intensités d'une même
commande.

Les têtes de récompense et de valeur ne produisent pas directement des scalaires. Avec
`support_size = 25`, elles retournent chacune `51` logits correspondant aux classes du support
`[-25, ..., 25]`. La tête de politique retourne seulement `2` logits, un par action.

Enfin, après `h` et chaque application de `g`, nous remettons le latent dans `[0,1]` par une
normalisation **min-max par échantillon**. Sans ce garde-fou, les applications répétées de la
dynamique pourraient faire dériver progressivement l'échelle du latent au cours d'une branche MCTS.
Cette normalisation stabilise les entrées reçues par les réseaux suivants, mais elle ne donne aucune
signification particulière aux dimensions latentes : c'est une astuce numérique, pas une hypothèse
sur la physique de CartPole.

In [ ]:
# Mini-test : reseaux h, g, f et latent borne
def scale_hidden_01(hidden_state):
    minimum = hidden_state.amin(dim=-1, keepdim=True)
    maximum = hidden_state.amax(dim=-1, keepdim=True)
    span = maximum - minimum
    safe_span = torch.where(span > 1e-8, span, torch.ones_like(span))
    scaled = (hidden_state - minimum) / safe_span
    if hidden_state.shape[-1] == 1:
        return torch.zeros_like(hidden_state)
    return scaled

def make_mlp(input_dim, hidden_dim, output_dim):
    return nn.Sequential(
        nn.Linear(input_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, hidden_dim),
        nn.SiLU(),
        nn.Linear(hidden_dim, output_dim),
    )

class Representation(nn.Module):
    def __init__(self, obs_dim, hidden_dim, latent_dim):
        super().__init__()
        self.net = make_mlp(obs_dim, hidden_dim, latent_dim)

    def forward(self, observation):
        return scale_hidden_01(self.net(observation))

class Dynamics(nn.Module):
    def __init__(self, latent_dim, num_actions, hidden_dim, support_size):
        super().__init__()
        self.num_actions = num_actions
        self.transition = make_mlp(latent_dim + num_actions, hidden_dim, latent_dim)
        self.reward_head = make_mlp(latent_dim, hidden_dim, 2 * support_size + 1)

    def forward(self, hidden_state, actions):
        action_one_hot = F.one_hot(actions.long(), num_classes=self.num_actions).float()
        x = torch.cat([scale_hidden_01(hidden_state), action_one_hot], dim=-1)
        next_hidden = scale_hidden_01(self.transition(x))
        reward_logits = self.reward_head(next_hidden)
        return next_hidden, reward_logits

class Prediction(nn.Module):
    def __init__(self, latent_dim, hidden_dim, num_actions, support_size):
        super().__init__()
        self.policy_head = make_mlp(latent_dim, hidden_dim, num_actions)
        self.value_head = make_mlp(latent_dim, hidden_dim, 2 * support_size + 1)

    def forward(self, hidden_state):
        hidden = scale_hidden_01(hidden_state)
        return self.policy_head(hidden), self.value_head(hidden)

_repr = Representation(obs_dim=4, hidden_dim=32, latent_dim=8)
_dyn = Dynamics(latent_dim=8, num_actions=2, hidden_dim=32, support_size=10)
_pred = Prediction(latent_dim=8, hidden_dim=32, num_actions=2, support_size=10)

sample_obs = torch.randn(5, 4)
hidden = _repr(sample_obs)
next_hidden, reward_logits = _dyn(hidden, torch.tensor([0, 1, 0, 1, 0]))
policy_logits, value_logits = _pred(next_hidden)

assert hidden.shape == (5, 8)
assert 0.0 <= float(hidden.min()) <= float(hidden.max()) <= 1.0
assert next_hidden.shape == (5, 8)
assert reward_logits.shape == (5, 21)
assert policy_logits.shape == (5, 2)
assert value_logits.shape == (5, 21)
print("h/g/f OK")

## II.7 — Assembler `h`, `g` et `f` dans `MuZeroAgent`

Les trois réseaux restent des briques indépendantes parce qu'ils répondent à trois questions
différentes. L'agent les rassemble toutefois derrière deux opérations qui correspondent exactement
au pseudocode MuZero :

- `initial_inference(o_t)` applique `h`, puis `f` à l'observation réelle ;
- `recurrent_inference(s^k, a^k)` applique `g`, puis `f` dans l'arbre imaginé.

Le MCTS, l'historique de partie et le replay restent hors de cette classe. Ils orchestrent le calcul ;
l'agent possède les fonctions apprises et sait les interroger. Cette séparation évite le dictionnaire
anonyme `{"h": ..., "g": ..., "f": ...}` sans cacher la recherche.

In [ ]:
class MuZeroAgent(nn.Module):
    def __init__(self, representation, dynamics, prediction, support_size, num_actions):
        super().__init__()
        self.representation = representation
        self.dynamics = dynamics
        self.prediction = prediction
        self.support_size = int(support_size)
        self.num_actions = int(num_actions)

    def initial_inference(self, observation):
        hidden_state = self.representation(observation)
        policy_logits, value_logits = self.prediction(hidden_state)
        return hidden_state, policy_logits, value_logits

    def recurrent_inference(self, hidden_state, action):
        next_hidden, reward_logits = self.dynamics(hidden_state, action)
        policy_logits, value_logits = self.prediction(next_hidden)
        return next_hidden, reward_logits, policy_logits, value_logits

    def loss(self, batch, config):
        return muzero_loss(self, batch, config)


_agent_smoke = MuZeroAgent(_repr, _dyn, _pred, support_size=10, num_actions=2)
_h0, _p0, _v0 = _agent_smoke.initial_inference(sample_obs)
_h1, _r1, _p1, _v1 = _agent_smoke.recurrent_inference(
    _h0, torch.tensor([0, 1, 0, 1, 0])
)
assert _h1.shape == _h0.shape
assert _p0.shape == _p1.shape == (5, 2)
assert _v0.shape == _v1.shape == _r1.shape == (5, 21)
print("MuZeroAgent : initial inference et recurrent inference vérifiées")

**Lecture.** Remarquez ce qui est délibérément absent : aucun décodeur d'observation, aucune tête
auxiliaire spectaculaire, aucune abstraction cachee. On veut que la forme du code rappelle directement
la décomposition du papier. Si vous comprenez ces trois modules, vous comprenez déjà la moitie de
MuZero.

In [ ]:
raw_latent = _repr.net(torch.tensor(np.stack([obs] * 4), dtype=torch.float32))
scaled_latent = scale_hidden_01(raw_latent)

fig, ax = plt.subplots(1, 2, figsize=(9.5, 3.6))
im0 = ax[0].imshow(raw_latent.detach().numpy(), aspect="auto", cmap="coolwarm")
ax[0].set_title("latent brut")
ax[0].set_xlabel("dimension latente")
ax[0].set_ylabel("exemple")
fig.colorbar(im0, ax=ax[0], fraction=0.046)

im1 = ax[1].imshow(scaled_latent.detach().numpy(), aspect="auto", cmap="viridis", vmin=0.0, vmax=1.0)
ax[1].set_title("latent remis dans [0,1]")
ax[1].set_xlabel("dimension latente")
ax[1].set_ylabel("exemple")
fig.colorbar(im1, ax=ax[1], fraction=0.046)
fig.tight_layout()
plt.show()

**Lecture.** Le min-max ne donne pas un "sens" interprétable aux dimensions latentes. Il fixe plutôt
une **plage numérique stable** pour que la dynamique et la prédiction travaillent sur des amplitudes
comparables d'un échantillon à l'autre. C'est une astuce de robustesse, pas une hypothèse semantique.

**Pseudo-code de haut niveau — représentation et cibles**

$$
\boxed{
\begin{aligned}
&\textbf{Self-play, à chaque pas } t : \\
&\quad s_t^0 = h(o_t) \\
&\quad \text{MCTS sur } s_t^0 \Rightarrow \text{visites cibles } \pi_t,\ \text{valeur de recherche } \nu_t \\
&\quad \text{jouer } a_t \sim \pi_t;\quad \text{enregistrer } (o_t,\, a_t,\, r_{t+1},\, \pi_t,\, \nu_t) \\
&\textbf{Pour apprendre, depuis } o_t : \\
&\quad s_t^0 = h(o_t) \\
&\quad \textbf{pour } k = 0 \dots K : \\
&\quad\quad p_t^k,\, v_t^k = f(s_t^k);\quad \text{comparer à } \pi_{t+k},\ z_{t+k} \\
&\quad\quad \textbf{si } k < K : \ s_t^{k+1},\, r_t^{k+1} = g(s_t^k,\, a_{t+k}) \\
\end{aligned}
}
$$

On voit déjà le cœur du contrat d'alignement : l'observation n'entre qu'au **début**. Tout le reste
se passe dans le latent.

---
# Partie III — MCTS MuZero from scratch

Nous avons maintenant les ingredients nécessaires pour coder la recherche elle-même. C'est la partie
la plus delicate conceptuellement, car plusieurs petites conventions s'y entremelent : bruit a la
racine, normalisation, récompense prédite par la dynamique, et inversion de perspective en deux
joueurs.

## III.1 — Nœuds et normalisation min-max

Un nœud MuZero stocke peu de choses :

- un **prior** ;
- un compteur de **visites** ;
- une somme de valeurs pour construire `Q` ;
- la **récompense immédiate** menant à ce nœud ;
- le **latent** associe ;
- le joueur `to_play` quand on est dans un jeu adversarial.

A côté, on garde un petit objet `MinMaxStats` qui observé les Q rencontrés pendant la recherche pour
les exprimer sur une échelle stable.

In [ ]:
# Mini-test visuel : construire un petit arbre et normaliser ses valeurs Q

class MinMaxStats:
    """Conserve les valeurs Q extrêmes rencontrées pendant une recherche."""

    def __init__(self):
        self.minimum = float("inf")
        self.maximum = float("-inf")

    def update(self, value):
        if math.isfinite(value):
            self.minimum = min(self.minimum, value)
            self.maximum = max(self.maximum, value)

    def normalize(self, value):
        if not math.isfinite(value):
            return 0.0
        if self.maximum <= self.minimum:
            return value
        return (value - self.minimum) / (self.maximum - self.minimum)

class Node:
    """Statistiques associées à un état latent de l'arbre MuZero."""

    def __init__(self, prior, to_play=1):
        self.prior = float(prior)
        self.to_play = int(to_play)
        self.visit_count = 0
        self.value_sum = 0.0
        self.reward = 0.0
        self.hidden_state = None
        self.children = {}

    def value(self):
        """Valeur moyenne des backups reçus par ce nœud."""
        return self.value_sum / self.visit_count if self.visit_count else 0.0

    def expanded(self):
        return bool(self.children)

def record_backups(node, backed_up_values):
    """Simule plusieurs backups traversant le même nœud."""
    for value in backed_up_values:
        node.visit_count += 1
        node.value_sum += float(value)

def edge_q(node, discount=0.997):
    """Valeur de l'arête menant au nœud : récompense immédiate + futur."""
    return node.reward + discount * node.value()


In [ ]:
# ------------------------------------------------------------------
# 1. Construction d'un petit arbre : racine -> 3 actions -> 2 suites
# ------------------------------------------------------------------

root = Node(prior=1.0)

child_specs = {
    0: {
        "prior": 0.50,
        "reward": 0.10,
        "backups": [0.90, 0.80, 0.70, 0.85, 0.75, 0.80],
    },
    1: {
        "prior": 0.30,
        "reward": 0.00,
        "backups": [0.55, 0.45, 0.40, 0.50],
    },
    2: {
        "prior": 0.20,
        "reward": -0.10,
        "backups": [0.25, 0.15],
    },
}

grandchild_specs = {
    0: [
        {"prior": 0.65, "reward": 0.15, "backups": [0.95, 0.85, 0.90]},
        {"prior": 0.35, "reward": 0.05, "backups": [0.55, 0.65]},
    ],
    1: [
        {"prior": 0.45, "reward": 0.00, "backups": [0.50, 0.40]},
        {"prior": 0.55, "reward": -0.05, "backups": [0.25, 0.35]},
    ],
    2: [
        {"prior": 0.70, "reward": -0.10, "backups": [0.20]},
        {"prior": 0.30, "reward": 0.00, "backups": [-0.10]},
    ],
}

for action, spec in child_specs.items():
    child = Node(prior=spec["prior"])
    child.reward = spec["reward"]
    record_backups(child, spec["backups"])
    root.children[action] = child

    for next_action, grandchild_spec in enumerate(grandchild_specs[action]):
        grandchild = Node(prior=grandchild_spec["prior"])
        grandchild.reward = grandchild_spec["reward"]
        record_backups(grandchild, grandchild_spec["backups"])
        child.children[next_action] = grandchild

# La racine a été traversée autant de fois que ses enfants réunis.
root.visit_count = sum(child.visit_count for child in root.children.values())
root.value_sum = sum(
    child.visit_count * edge_q(child)
    for child in root.children.values()
)


In [ ]:
# ------------------------------------------------------------------
# 2. Min-max sur toutes les arêtes déjà évaluées
# ------------------------------------------------------------------

mm = MinMaxStats()
evaluated_nodes = []

for child in root.children.values():
    evaluated_nodes.append(child)
    evaluated_nodes.extend(child.children.values())

for node in evaluated_nodes:
    mm.update(edge_q(node))

normalized_q = {
    id(node): mm.normalize(edge_q(node))
    for node in evaluated_nodes
}

# Vérifications comportementales
assert root.expanded()
assert root.visit_count == 12
assert root.children[0].visit_count > root.children[2].visit_count
assert edge_q(root.children[0]) > edge_q(root.children[1])
assert abs(mm.normalize(mm.minimum) - 0.0) < 1e-8
assert abs(mm.normalize(mm.maximum) - 1.0) < 1e-8
assert all(0.0 <= value <= 1.0 for value in normalized_q.values())


In [ ]:
# ------------------------------------------------------------------
# 3. Visualisation : couleur = Q normalisé, texte = P, N, Q
# ------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(12, 6))
cmap = plt.cm.viridis
norm = Normalize(vmin=0.0, vmax=1.0)

root_xy = (0.0, 0.0)
child_x = {0: -3.2, 1: 0.0, 2: 3.2}

ax.scatter(*root_xy, s=2300, color="#1d4ed8", edgecolor="black", zorder=3)
ax.text(
    *root_xy,
    f"racine\nN={root.visit_count}\nV={root.value():.2f}",
    ha="center",
    va="center",
    color="white",
    fontsize=10,
    fontweight="bold",
)

for action, child in root.children.items():
    cx, cy = child_x[action], -1.8
    q_value = edge_q(child)
    q_norm = normalized_q[id(child)]
    color = cmap(norm(q_norm))

    ax.plot([root_xy[0], cx], [root_xy[1], cy], color="#64748b", lw=2)
    ax.text(
        (root_xy[0] + cx) / 2,
        (root_xy[1] + cy) / 2 + 0.12,
        f"a={action}",
        ha="center",
        fontsize=9,
    )

    ax.scatter(cx, cy, s=1900, color=color, edgecolor="black", zorder=3)
    ax.text(
        cx,
        cy,
        f"a{action}\nP={child.prior:.2f}  N={child.visit_count}\n"
        f"Q={q_value:.2f}  Qnorm={q_norm:.2f}",
        ha="center",
        va="center",
        color="white" if q_norm < 0.55 else "black",
        fontsize=9,
    )

    offsets = [-0.85, 0.85]
    for next_action, grandchild in child.children.items():
        gx, gy = cx + offsets[next_action], -3.8
        grand_q = edge_q(grandchild)
        grand_norm = normalized_q[id(grandchild)]
        grand_color = cmap(norm(grand_norm))

        ax.plot([cx, gx], [cy, gy], color="#94a3b8", lw=1.5)
        ax.text(
            (cx + gx) / 2,
            (cy + gy) / 2 + 0.10,
            f"a={next_action}",
            ha="center",
            fontsize=8,
        )

        ax.scatter(gx, gy, s=1250, color=grand_color, edgecolor="black", zorder=3)
        ax.text(
            gx,
            gy,
            f"P={grandchild.prior:.2f}\nN={grandchild.visit_count}\n"
            f"Q={grand_q:.2f}",
            ha="center",
            va="center",
            color="white" if grand_norm < 0.55 else "black",
            fontsize=8,
        )

sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
sm.set_array([])
colorbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
colorbar.set_label(r"$Q_{\mathrm{norm}}$ dans l'arbre")

ax.set_title(
    "Un petit arbre MuZero : les backups construisent N et Q,\n"
    "puis MinMaxStats remet toutes les valeurs sur [0, 1]"
)
ax.set_xlim(-5.0, 5.0)
ax.set_ylim(-4.8, 0.9)
ax.axis("off")
plt.show()

print(
    f"Q_min={mm.minimum:.3f} | Q_max={mm.maximum:.3f} | "
    f"{root.visit_count} simulations représentées"
)


**Lecture.**  Chaque passage d'une simulation dans un nœud incrémente $N$ et ajoute une nouvelle
estimation à `value_sum`. La méthode `value()` transforme ensuite cette somme en moyenne. La quantité
utilisée pour comparer une branche depuis son parent est ici
$Q=r+\gamma V$ : elle combine la récompense immédiate de l'arête et la valeur future moyenne du
nœud enfant.
La couleur ne représente pas la valeur brute, mais $Q_{\mathrm{norm}}$. Le meilleur et le moins bon
$Q$ rencontrés dans l'arbre deviennent respectivement 1 et 0, tandis que les autres sont placés
entre les deux. Cette normalisation conserve donc l'ordre des branches tout en donnant à PUCT une
échelle comparable à celle de son bonus d'exploration.

Le point important est la séparation des rôles. Le nœud garde son histoire locale ;
`MinMaxStats` regarde l'arbre comme un tout pour mettre les Q sur une échelle comparable. Cette
séparation évite d'enfouir la normalisation dans un coin opaque du nœud lui-même.

## III.2 — Sélection, expansion et bruit de Dirichlet

Une simulation MuZero descend dans l'arbre en répétant deux opérations : **sélectionner** une branche
déjà connue, puis **développer** la première branche encore inconnue.

### 1. Sélection : suivre le meilleur compromis `Q + U`

Depuis la racine, MuZero choisit à chaque nœud l'enfant qui maximise :

$$
Q_{\mathrm{norm}}(s,a)+U(s,a).
$$

- $Q_{\mathrm{norm}}$ favorise les branches qui ont déjà produit de bonnes valeurs ;
- $U$ favorise les branches encore peu visitées, en tenant compte du prior $P(s,a)$.

La simulation continue ainsi jusqu'à rencontrer une arête dont l'enfant n'a pas encore été évalué.
Une même simulation peut donc exploiter une branche très prometteuse près de la racine, puis explorer
une action encore incertaine plus profondément dans l'arbre.

### 2. Expansion : transformer une feuille en nouveau sous-arbre

Lorsqu'une action non développée est atteinte, MuZero appelle sa dynamique apprise :

```text
latent parent + action
        ↓ g
nouveau latent + récompense prédite
        ↓ f
logits de politique + valeur de feuille
```

Les logits de politique sont transformés par `softmax` en priorités $P(s',a')$. MuZero crée alors un
enfant pour chaque action disponible. Chaque nouvel enfant commence avec :

- son prior $P(s',a')$ ;
- `visit_count = 0` ;
- `value_sum = 0` ;
- aucune représentation latente tant que cette arête n'est pas elle-même développée.

La valeur prédite à la feuille est ensuite remontée par le backup. L'expansion n'explore donc pas
immédiatement tous les enfants : elle prépare les possibilités que les simulations suivantes pourront
sélectionner.

### 3. Pourquoi perturber le prior à la racine ?

Au début de l'entraînement, la tête de politique est presque aléatoire. Pourtant, son prior influence
déjà les premières simulations du MCTS. Cela peut créer une boucle d'auto-confirmation :

```text
prior initial accidentellement élevé
        ↓
davantage de simulations dans cette branche
        ↓
davantage de visites à la racine
        ↓
ces visites deviennent la cible de politique
        ↓
le réseau renforce son prior initial
```

Sans exploration supplémentaire, une préférence née du hasard pourrait donc se renforcer simplement
parce qu'elle a reçu le premier budget de calcul.

MuZero perturbe les priorités **à la racine seulement** :

$$
P'(s_0,a)
=
(1-\eta)P(s_0,a)
+
\eta\,\xi_a,
\qquad
\boldsymbol{\xi}\sim\operatorname{Dir}(\alpha).
$$

### Qu'est-ce qu'une distribution de Dirichlet ?

La distribution de Dirichlet génère un **vecteur de probabilités** :

$$
\boldsymbol{\xi}=(\xi_1,\ldots,\xi_A),
\qquad
\xi_a\ge 0,
\qquad
\sum_{a=1}^{A}\xi_a=1.
$$

Elle ne produit donc pas un bruit indépendant pour chaque action. Elle produit directement une
nouvelle distribution valide sur l'ensemble des actions. C'est précisément ce qu'il faut pour
perturber un prior sans casser sa normalisation.

Le paramètre $\alpha$ contrôle la forme de cette distribution :

| Valeur de $\alpha$ | Forme des échantillons |
|---|---|
| $\alpha < 1$ | distributions souvent très asymétriques : quelques actions reçoivent beaucoup de masse |
| $\alpha = 1$ | distributions réparties uniformément sur le simplexe |
| $\alpha > 1$ | distributions concentrées autour d'une répartition presque uniforme |

Sur CartPole, nous utilisons $\alpha=0{,}25$. Comme il n'existe que deux actions, le bruit favorise
souvent temporairement l'une des deux directions. D'une recherche à l'autre, cette asymétrie change,
ce qui oblige l'agent à examiner les deux possibilités au cours du self-play.

### Exemple numérique

Supposons que le réseau propose :

```text
P(gauche) = 0.80
P(droite) = 0.20
```

et que le tirage de Dirichlet produise :

```text
ξ(gauche) = 0.10
ξ(droite) = 0.90
```

Avec une fraction de bruit $\eta=0{,}25$ :

$$
P'
=
0{,}75
\begin{bmatrix}
0{,}80\\
0{,}20
\end{bmatrix}
+
0{,}25
\begin{bmatrix}
0{,}10\\
0{,}90
\end{bmatrix}
=
\begin{bmatrix}
0{,}625\\
0{,}375
\end{bmatrix}.
$$

Le réseau conserve l'essentiel de son opinion : gauche reste prioritaire. Mais droite passe de `0.20`
à `0.375`, ce qui lui donne une chance réaliste de recevoir plusieurs simulations et éventuellement
de démontrer qu'elle est meilleure que prévu.

### Rôle de $\eta$ et $\alpha$

Les deux hyperparamètres ne jouent pas le même rôle :

- $\alpha$ contrôle la **forme** du bruit : diffus ou concentré sur quelques actions ;
- $\eta$ contrôle la **quantité** de bruit mélangée au prior du réseau.

Dans notre configuration, $\eta=0{,}25$ signifie que le prior final contient `75 %` de la proposition
du réseau et `25 %` d'exploration Dirichlet.

### Pourquoi uniquement à la racine ?

La racine correspond à la décision qui sera réellement exécutée. C'est donc ici que l'on veut
produire de la diversité entre les épisodes de self-play.

Injecter du bruit dans tous les nœuds rendrait chaque simulation artificiellement instable : une même
branche changerait de priorité simplement parce qu'elle est revisitée. À l'intérieur de l'arbre,
PUCT fournit déjà son propre mécanisme d'exploration. Le Dirichlet sert seulement à diversifier le
**point de départ réel** de la recherche.

Enfin, ce bruit est utilisé pendant la collecte d'expérience, mais désactivé pendant l'évaluation.
Lorsqu'on mesure la performance de l'agent, on veut observer sa meilleure décision actuelle, pas une
décision volontairement perturbée. Le générateur aléatoire doit aussi rester persistant : le recréer
avec la même seed à chaque recherche produirait toujours le même « bruit » et supprimerait la
diversité recherchée.

In [ ]:
# Mini-test visuel : masque légal, bruit de Dirichlet et décomposition du score PUCT

def ucb_score(parent, child, min_max_stats, config):
    # Coefficient d'exploration MuZero : il croît lentement avec les visites du parent.
    pb_c = (
        math.log(
            (parent.visit_count + config["pb_c_base"] + 1.0)
            / config["pb_c_base"]
        )
        + config["pb_c_init"]
    )
    pb_c *= math.sqrt(max(parent.visit_count, 1)) / (child.visit_count + 1)

    prior_score = pb_c * child.prior

    # Un enfant non visité ne possède pas encore de valeur fiable.
    if child.visit_count == 0:
        value_score = 0.0
    else:
        child_value = -child.value() if config["two_player"] else child.value()
        q_value = child.reward + config["discount"] * child_value
        value_score = min_max_stats.normalize(q_value)

    return value_score + prior_score


def select_child(parent, min_max_stats, config):
    """Sélectionne l'action de meilleur score Q_normalisé + U."""
    return max(
        sorted(parent.children.items()),
        key=lambda item: ucb_score(parent, item[1], min_max_stats, config),
    )


def expand_node(node, policy_logits, legal_actions=None, to_play=None):
    """Transforme les logits de politique en enfants munis de priorités légales."""
    priors = torch.softmax(
        policy_logits.reshape(-1), dim=0
    ).detach().cpu().numpy()

    # Les actions illégales sont annulées AVANT de renormaliser les priorités.
    if legal_actions is not None:
        mask = np.zeros_like(priors)
        mask[list(legal_actions)] = 1.0
        priors = priors * mask

    total = float(priors.sum())
    if total <= 0.0:
        # Fallback défensif : distribution uniforme sur les actions autorisées.
        if legal_actions is None:
            priors[:] = 1.0 / len(priors)
        else:
            priors[:] = 0.0
            priors[list(legal_actions)] = 1.0 / len(legal_actions)
    else:
        priors = priors / total

    child_to_play = node.to_play if to_play is None else to_play
    node.children = {
        action: Node(prior=prior, to_play=child_to_play)
        for action, prior in enumerate(priors)
        if prior > 0.0
    }


# RNG persistant : reproductible, mais le tirage change à chaque recherche.
_noise_rng = np.random.default_rng(0)


def add_exploration_noise(node, dirichlet_alpha, exploration_fraction):
    """Mélange le prior réseau avec un tirage de Dirichlet à la racine."""
    actions = sorted(node.children)
    noise = _noise_rng.dirichlet(
        [dirichlet_alpha] * len(actions)
    )

    for action, sample in zip(actions, noise):
        child = node.children[action]
        child.prior = (
            (1.0 - exploration_fraction) * child.prior
            + exploration_fraction * float(sample)
        )

    return np.asarray(noise, dtype=np.float32)


In [ ]:
cfg_search = {
    "pb_c_base": 19652.0,
    "pb_c_init": 1.25,
    "discount": 0.997,
    "dirichlet_alpha": 0.25,
    "exploration_fraction": 0.25,
    "two_player": False,
}

# ------------------------------------------------------------------
# 1. Expansion : logits -> softmax -> masque légal -> renormalisation
# ------------------------------------------------------------------

policy_logits = torch.tensor([2.0, 0.5, -1.0, 1.0])
raw_priors = torch.softmax(policy_logits, dim=0).numpy()

legal_actions = [0, 1, 3]  # l'action 2 est volontairement interdite
test_root = Node(prior=1.0)
expand_node(
    test_root,
    policy_logits,
    legal_actions=legal_actions,
    to_play=1,
)

actions = sorted(test_root.children)
priors_before = np.array(
    [test_root.children[action].prior for action in actions],
    dtype=np.float32,
)


In [ ]:
# ------------------------------------------------------------------
# 2. Quelques visites fictives donnent un Q différent à chaque enfant
# ------------------------------------------------------------------

backed_up_values = {
    0: [0.80, 0.70, 0.75, 0.65, 0.70, 0.72],
    1: [0.60, 0.50, 0.55],
    3: [0.20],
}
rewards = {0: 0.05, 1: 0.00, 3: -0.05}

for action, values in backed_up_values.items():
    child = test_root.children[action]
    child.reward = rewards[action]
    child.visit_count = len(values)
    child.value_sum = float(np.sum(values))

test_root.visit_count = sum(
    child.visit_count for child in test_root.children.values()
)

# MinMaxStats observe les Q des arêtes déjà évaluées.
search_stats = MinMaxStats()
q_values = {}

for action, child in test_root.children.items():
    q_value = (
        child.reward
        + cfg_search["discount"] * child.value()
    )
    q_values[action] = q_value
    search_stats.update(q_value)


In [ ]:
# ------------------------------------------------------------------
# 3. Bruit racine : 75 % prior réseau + 25 % Dirichlet
# ------------------------------------------------------------------

dirichlet_sample = add_exploration_noise(
    test_root,
    dirichlet_alpha=cfg_search["dirichlet_alpha"],
    exploration_fraction=cfg_search["exploration_fraction"],
)

priors_after = np.array(
    [test_root.children[action].prior for action in actions],
    dtype=np.float32,
)

# Décomposer chaque score en Q_normalisé + U.
q_normalized = []
exploration_bonuses = []
total_scores = []

for action in actions:
    child = test_root.children[action]
    q_norm = search_stats.normalize(q_values[action])
    score = ucb_score(test_root, child, search_stats, cfg_search)

    q_normalized.append(q_norm)
    exploration_bonuses.append(score - q_norm)
    total_scores.append(score)

q_normalized = np.asarray(q_normalized)
exploration_bonuses = np.asarray(exploration_bonuses)
total_scores = np.asarray(total_scores)

selected_action, _ = select_child(
    test_root, search_stats, cfg_search
)

# Vérifications comportementales
assert set(test_root.children) == {0, 1, 3}
assert raw_priors[2] > 0.0
assert np.isclose(priors_before.sum(), 1.0)
assert np.isclose(priors_after.sum(), 1.0)
assert np.isclose(dirichlet_sample.sum(), 1.0)
assert not np.allclose(priors_before, priors_after)
assert selected_action == actions[int(np.argmax(total_scores))]


In [ ]:
# ------------------------------------------------------------------
# 4. Visualisation des trois transformations
# ------------------------------------------------------------------

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Panneau 1 : effet du masque légal
all_actions = np.arange(len(raw_priors))
masked_priors = np.zeros_like(raw_priors)
for action, prior in zip(actions, priors_before):
    masked_priors[action] = prior

width = 0.36
axes[0].bar(
    all_actions - width / 2,
    raw_priors,
    width,
    label="softmax brut",
    color="#94a3b8",
)
axes[0].bar(
    all_actions + width / 2,
    masked_priors,
    width,
    label="après masque",
    color="#2563eb",
)
axes[0].axvspan(1.7, 2.3, color="#ef4444", alpha=0.12)
axes[0].set_xticks(all_actions, [f"a{i}" for i in all_actions])
axes[0].set_ylabel("probabilité")
axes[0].set_title("1. Masquer les actions illégales")
axes[0].legend()

# Panneau 2 : mélange avec le bruit de Dirichlet
x = np.arange(len(actions))
bar_width = 0.25

axes[1].bar(
    x - bar_width,
    priors_before,
    bar_width,
    label="prior réseau",
    color="#2563eb",
)
axes[1].bar(
    x,
    dirichlet_sample,
    bar_width,
    label="tirage Dirichlet",
    color="#f59e0b",
)
axes[1].bar(
    x + bar_width,
    priors_after,
    bar_width,
    label="prior mélangé",
    color="#10b981",
)
axes[1].set_xticks(x, [f"a{action}" for action in actions])
axes[1].set_ylabel("probabilité")
axes[1].set_title("2. Injecter le bruit à la racine")
axes[1].legend()

# Panneau 3 : le score final est Q_normalisé + U
axes[2].bar(
    x,
    q_normalized,
    label=r"$Q_{\mathrm{norm}}$",
    color="#0ea5e9",
)
axes[2].bar(
    x,
    exploration_bonuses,
    bottom=q_normalized,
    label=r"$U$",
    color="#f97316",
)

for index, score in enumerate(total_scores):
    axes[2].text(
        index,
        score + 0.03,
        f"{score:.2f}",
        ha="center",
        fontweight="bold",
    )

selected_index = actions.index(selected_action)
axes[2].scatter(
    selected_index,
    total_scores[selected_index] + 0.13,
    marker="*",
    s=220,
    color="#16a34a",
    edgecolor="black",
    label=f"action choisie : a{selected_action}",
    zorder=4,
)

axes[2].set_xticks(x, [f"a{action}" for action in actions])
axes[2].set_ylabel("score PUCT")
axes[2].set_title("3. Sélectionner avec $Q_{norm}+U$")
axes[2].legend()

fig.suptitle(
    "Une décision PUCT complète : logits → masque → Dirichlet → sélection",
    fontsize=13,
)
fig.tight_layout()
plt.show()


**Lecture**. Le premier panneau montre que le masque intervient avant toute recherche : l'action
a2 possédait une probabilité non nulle dans le softmax du réseau, mais elle disparaît complètement
après l'application des contraintes légales.
Le deuxième panneau sépare les trois distributions qu'il ne faut pas confondre. Le prior réseau
représente l'opinion actuelle de la politique, le tirage de Dirichlet propose une direction
d'exploration aléatoire, et le prior mélangé est celui qui sera réellement utilisé à la racine
pendant le self-play.
Enfin, le troisième panneau montre que PUCT ne sélectionne pas nécessairement l'action de plus grand
prior. Chaque branche reçoit un score composé de sa qualité déjà observée
$Q_{\mathrm{norm}}$ et de son bonus d'exploration $U$. L'étoile indique l'action effectivement
choisie pour la prochaine simulation.Deux détails méritent d'être gardes très nets en tête. D'abord, le masque d'actions
légales agit **avant** la normalisation des priorités. Ensuite, le bruit de Dirichlet ne doit pas
fuiter partout dans l'arbre : il sert à desserrer la racine, pas à brouiller toutes les estimations.

## III.3 — Backup, discount et inversion de perspective

Après la sélection et l'expansion, le réseau évalue la nouvelle feuille. Cette valeur ne doit pas
rester uniquement dans la feuille : elle doit informer toutes les décisions qui ont conduit jusqu'à
elle. Le **backup** remonte donc la récompense et la valeur le long du chemin de recherche.

### Cas mono-joueur : une seule perspective

Dans CartPole, toutes les valeurs sont exprimées du point de vue du même agent. Si le futur situé un
niveau plus bas vaut $G_{k+1}$, alors le retour du nœud courant vaut :

$$
G_k=r_{k+1}+\gamma G_{k+1}.
$$

- $r_{k+1}$ est la récompense prédite pour la transition choisie ;
- $G_{k+1}$ est la valeur future remontée depuis l'enfant ;
- $\gamma$ réduit l'importance des récompenses lointaines.

Supposons qu'une action rapporte immédiatement `1`, puis conduise à une feuille de valeur `8`, avec
$\gamma=0{,}9$ :

$$
G_k=1+0{,}9\times8=8{,}2.
$$

Cette valeur est ajoutée à `value_sum`, puis le compteur de visites est incrémenté. Le nœud conserve
ainsi une moyenne de toutes les simulations qui l'ont traversé :

$$
N\leftarrow N+1,
\qquad
W\leftarrow W+G_k,
\qquad
Q=\frac{W}{N}.
$$

Le backup ne remplace donc pas l'ancienne estimation par la dernière simulation : il l'intègre à une
moyenne progressivement plus stable.

### Cas deux joueurs : la perspective change à chaque niveau

Dans Connect-4, une valeur n'est jamais « bonne » ou « mauvaise » dans l'absolu. Elle est bonne
**pour un joueur donné**.

Si une feuille vaut `+0.8` pour le joueur A, cette même situation vaut `-0.8` pour le joueur B. Or les
joueurs alternent à chaque profondeur de l'arbre. Lorsque l'on remonte d'un enfant vers son parent,
il faut donc changer de perspective.

Avec la convention utilisée ici, $G_{k+1}$ est exprimé du point de vue du joueur au trait dans
l'enfant. Le parent appartient à l'adversaire, donc :

$$
G_k=r_{k+1}-\gamma G_{k+1}.
$$

Le signe négatif ne signifie pas que la récompense est forcément négative. Il traduit uniquement le
changement de perspective : ce qui est favorable au joueur de l'enfant est défavorable au joueur du
parent.

### Exemple sans récompense intermédiaire

Dans Connect-4, les récompenses intermédiaires sont généralement nulles. Prenons $\gamma=1$ et une
feuille évaluée à `+0.8` pour le joueur A :

```mermaid
flowchart BT
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    leaf["Feuille<br/>joueur A au trait<br/>$$~G_A^{(2)}=+0.8$$"]
    child["Nœud enfant<br/>joueur B au trait<br/>$$~G_B^{(1)}=0-1\times(+0.8)=-0.8$$"]
    root["Racine<br/>joueur A au trait<br/>$$~G_A^{(0)}=0-1\times(-0.8)=+0.8$$"]

    leaf -->|"backup : perspective A vers B"| child
    child -->|"backup : perspective B vers A"| root

    classDef playerA fill:none,stroke:#2563eb,stroke-width:2px
    classDef playerB fill:none,stroke:#dc2626,stroke-width:2px

    class leaf,root playerA
    class child playerB
```

Les signes du diagramme sont donc bien cohérents :

1. la feuille vaut `+0.8` pour A ;
2. vue par B au niveau précédent, elle vaut `-0.8` ;
3. vue à nouveau par A à la racine, elle vaut `+0.8`.

Après deux changements de joueur, on retrouve logiquement la perspective initiale.

### Où le signe doit-il être appliqué ?

Cette convention doit être respectée dans trois endroits différents :

| Étape | Convention nécessaire |
|---|---|
| **Backup** | appliquer $r-\gamma G$ quand on remonte vers l'adversaire |
| **Sélection PUCT** | négativer la valeur d'un enfant avant de la comparer depuis son parent |
| **Cibles d'entraînement** | exprimer la valeur finale dans la perspective du joueur au trait à chaque position |

Par exemple, pendant la sélection, si `child.value()` est stockée dans la perspective du joueur de
l'enfant, le parent doit lire :

$$
Q_{\text{parent}}(s,a)
=
r(s,a)-\gamma\,V_{\text{enfant}}.
$$

Oublier ce signe dans un seul de ces trois endroits rend l'arbre incohérent. Le MCTS pourrait alors
favoriser une branche parce qu'elle est excellente pour l'adversaire — autrement dit, il traiterait
son adversaire comme un allié.

En mono-joueur, on utilise donc :

$$
G\leftarrow r+\gamma G.
$$

En deux joueurs à somme nulle, avec des valeurs stockées dans la perspective du joueur au trait :

$$
G\leftarrow r-\gamma G.
$$

La différence tient dans un seul signe, mais ce signe porte toute la logique adversariale du MCTS.

In [ ]:
# Mini-test : backup mono-joueur et deux joueurs
def backpropagate(search_path, value, discount, min_max_stats, two_player):
    current = float(value)
    for node in reversed(search_path):
        node.value_sum += current
        node.visit_count += 1
        if two_player:
            min_max_stats.update(node.reward - discount * node.value())   # même convention de signe que la sélection
            current = node.reward - discount * current
        else:
            min_max_stats.update(node.reward + discount * node.value())
            current = node.reward + discount * current

mm_single = MinMaxStats()
a = Node(1.0)
b = Node(1.0)
b.reward = 0.5
backpropagate([a, b], value=1.0, discount=0.9, min_max_stats=mm_single, two_player=False)
assert abs(b.value() - 1.0) < 1e-6
assert abs(a.value() - 1.4) < 1e-6

mm_two = MinMaxStats()
c = Node(1.0, to_play=1)
d = Node(1.0, to_play=-1)
d.reward = 0.25
backpropagate([c, d], value=1.0, discount=1.0, min_max_stats=mm_two, two_player=True)
assert abs(d.value() - 1.0) < 1e-6
assert abs(c.value() - (-0.75)) < 1e-6
print("backup OK")

**Lecture.** Le test deux joueurs est le genre de détail qu'on croit mineur avant de se tromper en
implementation. Sans inversion de perspective, le MCTS traite l'adversaire comme un allie et la
recherche devient incoherente. C'est une erreur classique, et l'une des raisons pour lesquelles une
petite batterie de mini-tests vaut tellement de temps gagne.

In [ ]:
def run_mcts(observation, agent, config, legal_actions=None, to_play=1, add_root_noise=True):
    num_actions = agent.num_actions
    support_size = agent.support_size
    root = Node(prior=1.0, to_play=to_play)

    with torch.no_grad():
        hidden, policy_logits, value_logits = agent.initial_inference(observation.unsqueeze(0))
    root.hidden_state = hidden
    root_value = float(support_to_scalar(value_logits, support_size)[0])
    expand_node(root, policy_logits[0], legal_actions=legal_actions, to_play=to_play if not config["two_player"] else -to_play)

    if add_root_noise:
        add_exploration_noise(root, config["dirichlet_alpha"], config["exploration_fraction"])

    min_max_stats = MinMaxStats()
    for _ in range(config["num_simulations"]):
        node = root
        search_path = [node]
        actions = []

        while node.expanded():
            action, node = select_child(search_path[-1], min_max_stats, config)
            search_path.append(node)
            actions.append(action)
            if not node.expanded():
                break

        parent = search_path[-2]
        action_tensor = torch.tensor([actions[-1]])
        with torch.no_grad():
            next_hidden, reward_logits, policy_logits, value_logits = agent.recurrent_inference(
                parent.hidden_state, action_tensor
            )
        node.hidden_state = next_hidden
        node.reward = float(support_to_scalar(reward_logits, support_size)[0])
        node.to_play = parent.to_play if not config["two_player"] else -parent.to_play
        expand_node(node, policy_logits[0], legal_actions=None, to_play=node.to_play if not config["two_player"] else -node.to_play)
        value = float(support_to_scalar(value_logits, support_size)[0])
        backpropagate(search_path, value, config["discount"], min_max_stats, config["two_player"])

    root_value = root.value()  # F1 : valeur de RECHERCHE (et non la valeur brute du réseau)
    visits = np.zeros(num_actions, dtype=np.float32)
    priors = np.zeros(num_actions, dtype=np.float32)
    q_values = np.zeros(num_actions, dtype=np.float32)
    for action, child in root.children.items():
        visits[action] = child.visit_count
        priors[action] = child.prior
        q_values[action] = child.reward + config["discount"] * child.value()
    if visits.sum() > 0:
        visits = visits / visits.sum()
    return root, root_value, priors, visits, q_values

**Lecture.** La structuré est simple à lire une fois mise a plat. `h` intervient une seule fois a la
racine ; ensuite `g` et `f` alternent pour avancer dans le latent et reevaluer la feuille. La
recherche n'a donc besoin ni des observations futures, ni d'un simulateur externe : elle vit
entierement dans son monde compact.

In [ ]:
search_agent = MuZeroAgent(_repr, _dyn, _pred, support_size=10, num_actions=2)
search_cfg = {
    "num_simulations": 30,
    "discount": 0.997,
    "pb_c_base": 19652.0,
    "pb_c_init": 1.25,
    "dirichlet_alpha": 0.25,
    "exploration_fraction": 0.25,
    "two_player": False,
    "noise_seed": 5,
}
obs_tensor = torch.tensor(obs, dtype=torch.float32)
root, root_value, priors, visits, q_values = run_mcts(obs_tensor, search_agent, search_cfg)

x = np.arange(search_agent.num_actions)
plt.figure(figsize=(6.4, 3.6))
plt.bar(x - 0.2, priors, width=0.4, label="prior reseau")
plt.bar(x + 0.2, visits, width=0.4, label="visites MCTS")
plt.xticks(x, ["gauche", "droite"])
plt.ylabel("masse")
plt.title("Avant entrainement : le MCTS repond deja a ce que le prior suggere")
plt.legend()
plt.show()
print("root value:", round(root_value, 3), "| q estimates:", np.round(q_values, 3))

**Lecture.** Même avec des réseaux non entraînés, la logique structurelle est la bonne : le prior
propose une repartition initiale, puis les visites la reforment selon ce que la dynamique et la valeur
de feuille racontent. Une fois l'apprentissage lance, cet ecart entre prior et visites devient
justement le signal d'imitation de la tête de politique.

In [ ]:
no_noise_cfg = dict(search_cfg)
no_noise_cfg["noise_seed"] = 0
root_clean, _, priors_clean, visits_clean, _ = run_mcts(obs_tensor, search_agent, no_noise_cfg, add_root_noise=False)
root_noisy, _, priors_noisy, visits_noisy, _ = run_mcts(obs_tensor, search_agent, search_cfg, add_root_noise=True)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
ax[0].bar(x - 0.2, priors_clean, 0.4, label="prior")
ax[0].bar(x + 0.2, visits_clean, 0.4, label="visites")
ax[0].set_title("Sans bruit de Dirichlet")
ax[0].set_xticks(x, ["gauche", "droite"])
ax[0].legend()

ax[1].bar(x - 0.2, priors_noisy, 0.4, label="prior bruite")
ax[1].bar(x + 0.2, visits_noisy, 0.4, label="visites")
ax[1].set_title("Avec bruit de Dirichlet")
ax[1].set_xticks(x, ["gauche", "droite"])
ax[1].legend()

fig.suptitle("Le bruit racine evite une adhesion trop precoce au prior")
fig.tight_layout()
plt.show()

**Lecture.** Le bruit ne sert pas à "faire du hasard" pour le plaisir. Il sert à casser une boucle
d'auto-confirmation potentielle : si le prior initial est un peu faux, un MCTS trop docile peut lui
laisser trop de pouvoir trop vite. Le Dirichlet redonne une petite respiration exploratoire là où elle
compte le plus : au point de décision réel.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.4))
ax.scatter([0], [0], s=1500, color="#1d4ed8")
ax.text(0, 0, "root", color="white", ha="center", va="center")

for idx, action in enumerate(sorted(root.children)):
    child = root.children[action]
    child_x = -1 + 2 * idx
    ax.plot([0, child_x], [0, -1], color="#64748b")
    ax.scatter([child_x], [-1], s=1100, color="#0ea5e9")
    ax.text(
        child_x,
        -1,
        f"a{action}\nprior={child.prior:.2f}\nN={child.visit_count}",
        ha="center",
        va="center",
        color="white",
        fontsize=9,
    )
    grand_children = list(sorted(child.children.items()))[:2]
    for j, (ga, gc) in enumerate(grand_children):
        gx = child_x + (-0.4 if j == 0 else 0.4)
        ax.plot([child_x, gx], [-1, -2], color="#94a3b8")
        ax.scatter([gx], [-2], s=650, color="#22c55e")
        ax.text(gx, -2, f"{ga}\nN={gc.visit_count}", ha="center", va="center", fontsize=8)

ax.set_title("Petit arbre MuZero : prior a la racine, visites apres recherche")
ax.axis("off")
plt.show()

**Lecture.** Ce mini-arbre rend visible la phrase "le réseau propose, la recherche dispose". Le prior
est stocke dans les enfants de la racine, mais les compteurs de visites apparaissent seulement après
les simulations. Autrement dit, la politique finale d'action n'est pas directement `softmax(policy)` :
elle est **médiée par le calcul de recherche**.

**Pseudo-code de la recherche MuZero**

$$
\boxed{
\begin{aligned}
&s^0 = h(o);\quad \text{racine} = \text{expand}(f(s^0));\quad \text{bruit de Dirichlet à la racine} \\
&\textbf{répéter } N \textbf{ simulations :} \\
&\quad \text{descendre via } \arg\max_a \big(Q_{\text{norm}}(s,a) + U(s,a)\big) \text{ jusqu'à une feuille} \\
&\quad s',\, r = g(s, a);\quad p,\, v = f(s');\quad \text{expand(feuille, } p) \\
&\quad \text{backup}(v) \text{ avec récompenses et discount} \\
&\quad \text{(deux joueurs : inverser la perspective à chaque niveau)} \\
&\pi = \text{visites des enfants de la racine, normalisées} \\
\end{aligned}
}
$$

C'est cette politique de visites qui servira à l'entraînement, pas seulement le prior brut du réseau.

---
# Partie IV — Self-play, replay et apprentissage CartPole

La recherche seule ne suffit pas. Il faut maintenant transformer les épisodes en exemples
d'apprentissage : observations d'origine, actions exécutées, récompenses réelles, valeurs racine,
et politiques de visites. C'est le pont entre "penser" et "apprendre a mieux penser".

## IV.1 — Historique de partie et cibles n-step

Une partie de self-play ne stocke pas seulement les transitions de l'environnement. Pour chaque
position $t$, MuZero conserve aussi le résultat de la recherche effectuée avant de choisir l'action :

- l'observation réelle $o_t$ ;
- l'action exécutée $a_t$ ;
- la récompense obtenue après cette action, $r_{t+1}$ ;
- la valeur de recherche à la racine, $\nu_t$ ;
- la distribution normalisée des visites, $\pi_t$ ;
- le joueur au trait dans le cas adversarial.

On peut résumer une position par :

```text
(o_t, a_t, r_{t+1}, ν_t, π_t, to_play_t)
```

La distribution $\pi_t$ ne correspond pas directement au prior prédit par le réseau. Elle provient
des compteurs de visites du MCTS :

$$
\pi_t(a)=\frac{N(s_t^0,a)}{\sum_bN(s_t^0,b)}.
$$

Elle constitue donc une politique **améliorée par la recherche**, que la tête de politique tentera
ensuite d'imiter.

### Construire la cible de valeur

Pour entraîner la tête de valeur au temps $t$, MuZero utilise un **retour n-step bootstrappé** :

$$
z_t
=
\sum_{j=0}^{n-1}\gamma^j r_{t+1+j}
+
\gamma^n\nu_{t+n}.
$$

La première partie contient des récompenses réellement observées dans l'environnement. La seconde
utilise la valeur de recherche obtenue $n$ positions plus loin.

Avec $n=3$, la cible devient par exemple :

$$
z_t
=
r_{t+1}
+\gamma r_{t+2}
+\gamma^2r_{t+3}
+\gamma^3\nu_{t+3}.
$$

Cette cible réalise un compromis :

- un petit $n$ repose rapidement sur le bootstrap $\nu_{t+n}$ : la cible est stable mais dépend
  fortement des estimations actuelles du modèle ;
- un grand $n$ utilise davantage de récompenses réelles : la cible est moins biaisée par le réseau,
  mais plus variable et plus sensible aux longues trajectoires.

### Trois valeurs à ne pas confondre

| Quantité | Origine | Rôle |
|---|---|---|
| $v_t=f(h(o_t))$ | tête de valeur du réseau, avant recherche | première estimation de la racine |
| $\nu_t$ | moyenne des backups du MCTS à la racine | estimation améliorée par la recherche et stockée dans le replay |
| $z_t$ | récompenses réelles proches + bootstrap sur $\nu_{t+n}$ | cible utilisée pour entraîner la tête de valeur |

La distinction est importante : stocker $v_t$ à la place de $\nu_t$ reviendrait à apprendre au
réseau à reproduire directement sa propre estimation. En stockant la valeur de recherche, le réseau
apprend au contraire à imiter une estimation enrichie par plusieurs simulations.

### Fin d'épisode et états absorbants

Si l'épisode se termine avant $t+n$, il n'existe aucune valeur future sur laquelle bootstrapper. La
somme s'arrête à la dernière récompense réelle et le terme $\gamma^n\nu_{t+n}$ disparaît.

Lorsque le déroulage d'entraînement continue au-delà de la fin de la trajectoire, on utilise des
positions **absorbantes** :

- récompense cible nulle ;
- valeur cible nulle ;
- perte de politique masquée.

La politique est masquée parce qu'aucune action réelle n'a été recherchée dans ces états fictifs.
Lui imposer une distribution uniforme créerait une fausse cible et entraînerait inutilement la tête
de politique.

Enfin, dans un jeu à deux joueurs, $z_t$ doit toujours être exprimé dans la perspective du joueur au
trait à la position $t$. Une victoire pour le joueur A produit donc une cible positive aux positions
de A et négative aux positions de son adversaire. Cette convention doit rester identique dans le
replay, le backup et la sélection PUCT.

In [ ]:
# Mini-test : GameHistory, ReplayBuffer, make_target
class GameHistory:
    def __init__(self):
        self.observations = []
        self.actions = []
        self.rewards = []
        self.root_values = []
        self.child_visits = []
        self.to_play = []

    def append(self, obs, action, reward, root_value, visits, to_play=1):
        self.observations.append(np.asarray(obs, dtype=np.float32))
        self.actions.append(int(action))
        self.rewards.append(float(reward))
        self.root_values.append(float(root_value))
        self.child_visits.append(np.asarray(visits, dtype=np.float32))
        self.to_play.append(int(to_play))

    def __len__(self):
        return len(self.actions)

def make_target(game, start_index, num_unroll_steps, td_steps, discount, num_actions):
    # Renvoie AUSSI un masque de politique : 0 sur les pas APRÈS la fin de partie
    # (états absorbants), pour ne PAS entraîner la tête de politique vers l'uniforme.
    # La cible de valeur est exprimée dans la perspective du joueur au trait (to_play) :
    # sans effet en mono-joueur, signe inversé pour l'adversaire en jeu à deux.
    value_targets, reward_targets, policy_targets, policy_masks = [], [], [], []
    horizon = len(game)
    for k in range(num_unroll_steps + 1):
        index = start_index + k
        if index < horizon:
            perspective = game.to_play[index]
            bootstrap_index = index + td_steps
            value = 0.0
            for j in range(td_steps):
                if index + j < horizon:
                    sign = 1.0 if game.to_play[index + j] == perspective else -1.0
                    value += sign * (discount ** j) * game.rewards[index + j]
            if bootstrap_index < horizon:
                sign = 1.0 if game.to_play[bootstrap_index] == perspective else -1.0
                value += sign * (discount ** td_steps) * game.root_values[bootstrap_index]
            reward_targets.append(game.rewards[index - 1] if index > 0 else 0.0)
            policy_targets.append(game.child_visits[index])
            value_targets.append(value)
            policy_masks.append(1.0)
        else:
            reward_targets.append(0.0)
            policy_targets.append(np.zeros(num_actions, dtype=np.float32))
            value_targets.append(0.0)
            policy_masks.append(0.0)
    return value_targets, reward_targets, policy_targets, policy_masks

class ReplayBuffer:
    def __init__(self, capacity, seed=0):
        self.capacity = int(capacity)
        self.games = []
        self.rng = np.random.default_rng(seed)

    def save(self, game):
        self.games.append(game)
        if len(self.games) > self.capacity:
            self.games.pop(0)

    def sample_batch(self, batch_size, num_unroll_steps, td_steps, discount, num_actions):
        # Échantillonnage uniforme des POSITIONS : on tire la partie proportionnellement
        # à sa longueur (sinon une partie courte pèse autant qu'une longue).
        lengths = np.array([len(g) for g in self.games], dtype=np.float64)
        probs = lengths / lengths.sum()
        batch = []
        for _ in range(batch_size):
            game = self.games[int(self.rng.choice(len(self.games), p=probs))]
            start = int(self.rng.integers(len(game)))
            actions = game.actions[start : start + num_unroll_steps]
            if len(actions) < num_unroll_steps:
                pad = [int(self.rng.integers(num_actions)) for _ in range(num_unroll_steps - len(actions))]
                actions = actions + pad
            v, r, p, m = make_target(game, start, num_unroll_steps, td_steps, discount, num_actions)
            batch.append((game.observations[start], np.asarray(actions), np.asarray(v),
                          np.asarray(r), np.asarray(p), np.asarray(m)))
        return batch

toy_game = GameHistory()
toy_game.append([0, 0, 0, 0], 0, 1.0, 2.0, [0.8, 0.2])
toy_game.append([1, 0, 0, 0], 1, 1.0, 1.5, [0.3, 0.7])
toy_game.append([2, 0, 0, 0], 0, 1.0, 0.5, [0.6, 0.4])
v_t, r_t, p_t, m_t = make_target(toy_game, 0, num_unroll_steps=2, td_steps=2, discount=0.9, num_actions=2)
rb = ReplayBuffer(capacity=4, seed=1)
rb.save(toy_game)
batch = rb.sample_batch(1, num_unroll_steps=2, td_steps=2, discount=0.9, num_actions=2)
assert len(v_t) == 3 and len(r_t) == 3 and len(p_t) == 3 and len(m_t) == 3
assert batch[0][1].shape == (2,) and batch[0][5].shape == (3,)
print("targets/replay OK | masque =", m_t)


**Lecture.** L'alignement des indices est le piège central ici. A l'état `s_t^0`, la récompense cible
vaut `0` parce qu'elle correspond à la transition **menant** à cet état. Ensuite, au pas de déroulage
`k=1`, on prédit bien `r_{t+1}`. Cette convention à l'air bureaucratique, mais si on la rate, la
dynamique apprend a prédire la récompense du mauvais pas.

## IV.2 — Lire concrètement une cible n-step

L'équation de $z_t$ devient plus intuitive si on la sépare en deux morceaux :

1. des **récompenses proches**, connues parce qu'elles ont réellement été observées ;
2. une **valeur lointaine**, estimée par le MCTS à une position future.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    current["position actuelle<br/>$$~o_t$$"]
    r1["récompense réelle<br/>$$~r_{t+1}$$"]
    r2["récompense réelle<br/>$$~\gamma r_{t+2}$$"]
    rn["récompense réelle<br/>$$~\gamma^{n-1}r_{t+n}$$"]
    future["valeur de recherche future<br/>$$~\gamma^n\nu_{t+n}$$"]

    current --> r1 --> r2 -->|"..."| rn --> future
```

La cible complète est :

$$
z_t
=
r_{t+1}
+\gamma r_{t+2}
+\cdots
+\gamma^{n-1}r_{t+n}
+\gamma^n\nu_{t+n}.
$$

### Exemple : calculer une cible n-step sur CartPole

Supposons que :

- `td_steps = 3` ;
- $\gamma=0{,}997$ ;
- CartPole survit pendant les trois transitions, donc
  $r_{t+1}=r_{t+2}=r_{t+3}=1$ ;
- la recherche estime qu'à la position $t+3$, le futur vaut
  $\nu_{t+3}=80$.

La cible vaut alors :

$$
\begin{aligned}
z_t
&=
1
+0{,}997\times1
+0{,}997^2\times1
+0{,}997^3\times80\\
&\approx
1+0{,}997+0{,}994+79{,}282\\
&\approx82{,}27.
\end{aligned}
$$

Les trois premières contributions proviennent du **monde réel**. La dernière résume tout ce qui
pourrait arriver après ces trois pas, selon la recherche menée à la position future.

Le bootstrap évite d'attendre la fin complète de l'épisode pour construire une cible. Sans lui, une
position située au début d'un épisode de 500 pas ne pourrait être correctement évaluée qu'après avoir
observé toute la trajectoire.

### Que se passe-t-il si l'épisode se termine ?

Supposons maintenant que le poteau tombe après la deuxième transition. On observe seulement
$r_{t+1}$ et $r_{t+2}$ ; la position $t+3$ n'existe pas. La cible devient :

$$
z_t=r_{t+1}+\gamma r_{t+2}.
$$

On ne bootstrappe pas au-delà de la terminaison. La fin réelle de l'épisode coupe donc naturellement
la somme.

### Le compromis contrôlé par `td_steps`

Le paramètre `td_steps`, noté $n$ dans l'équation, détermine l'endroit où l'on cesse d'utiliser les
récompenses réelles pour faire confiance à la valeur de recherche.

- **Petit `td_steps`** : le bootstrap arrive rapidement. La cible est moins variable, mais dépend
  davantage de la qualité actuelle du réseau et du MCTS.
- **Grand `td_steps`** : davantage de récompenses réelles entrent dans la cible. Le biais de
  bootstrap diminue, mais la cible devient plus sensible aux variations de trajectoire et aux
  terminaisons.

Il ne faut pas confondre `td_steps` avec `num_unroll_steps` :

| Paramètre | Question contrôlée |
|---|---|
| `td_steps = n` | combien de récompenses réelles utilise-t-on avant le bootstrap ? |
| `num_unroll_steps = K` | sur combien d'actions déroule-t-on `g` pendant l'entraînement ? |

Dans notre configuration, on peut par exemple entraîner le modèle sur $K=5$ transitions latentes,
tout en construisant chaque cible de valeur avec $n=10$ récompenses futures. Ces deux horizons ont des
rôles différents : l'un contrôle la profondeur d'apprentissage du modèle, l'autre la profondeur de la
cible temporelle.

In [ ]:
value_targets, reward_targets, policy_targets, policy_masks = make_target(
    toy_game,
    start_index=0,
    num_unroll_steps=2,
    td_steps=2,
    discount=0.9,
    num_actions=2,
)

contributions = np.array([1.0, 0.9, 0.9 ** 2 * toy_game.root_values[2]])
labels = ["r1", "0.9 * r2", "0.9^2 * v2"]

plt.figure(figsize=(7.5, 3.6))
plt.bar(range(len(contributions)), contributions, color=["#22c55e", "#10b981", "#2563eb"])
plt.xticks(range(len(contributions)), labels)
plt.ylabel("contribution")
plt.title(f"Decomposition de z0 = {value_targets[0]:.3f}")
plt.show()
print("reward targets:", reward_targets)
print("policy target t0:", np.round(policy_targets[0], 3))

**Lecture.** Cette décomposition rend l'équation moins abstraite. `z_t` n'est ni un Monte-Carlo pur,
ni une simple prédiction réseau. C'est une couture entre le réel proche et le modèle lointain. Dans
MuZero, cette couture est particulierement naturelle parce que la valeur lointaine vient elle-même
d'une **recherche** dans le latent.

## IV.3 — Loss MuZero : politique, valeur, récompense

A chaque état déroule, on paie trois termes :

- une perte **politique** pour rapprocher les logits du réseau des visites MCTS ;
- une perte **valeur** pour prédire `z_t` ;
- une perte **récompense** pour prédire la récompense immédiate du modèle dynamique.

Le tout est somme sur `k = 0..K` le long du déroulage latent. On garde ici une version compacte, mais
l'idée est fidèle au papier : chaque pas latent doit rester cohérent comme lieu de prédiction.

**La perte se paie à chaque pas déroulé :**

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    s0["$$~s^0 = h(o_t)$$"]
    l0["pertes pas 0<br/>$$~\pi,\ v$$"]
    s1["$$~s^1 = g(s^0, a_t)$$"]
    l1["pertes pas 1<br/>$$~\pi,\ v,\ r$$"]
    sk["$$~s^K$$"]
    lk["pertes pas K<br/>$$~\pi,\ v,\ r$$"]
    s0 --> l0
    s0 --> s1 --> l1
    s1 -->|"..."| sk --> lk
```

Formellement, la perte moyennée sur les $K+1$ pas dépliés :

$$
\mathcal{L}(\theta) = \frac{1}{K+1}\sum_{k=0}^{K}\Big[\underbrace{\ell^{p}(\pi_{t+k},\, p_t^k)}_{\text{politique}} + \underbrace{\ell^{v}(z_{t+k},\, v_t^k)}_{\text{valeur}} + \underbrace{\ell^{r}(r_{t+k},\, \hat r_t^k)}_{\text{récompense}}\Big].
$$

Deux facteurs d'échelle viennent du papier : on **moyenne sur les $K+1$ pas** (le facteur $1/K$, sinon la magnitude dépendrait de la longueur de déroulage), et on **divise par deux le gradient** à l'entrée de chaque pas de dynamique `g` (la dynamique, sollicitée à tous les niveaux, ne doit pas recevoir un gradient cumulé démesuré). La perte de politique est en outre **masquée** au-delà de la fin de partie.

In [ ]:
# Mini-test : loss, gradients finis et batch minimal
def ce_support(logits, target_scalar, support_size):
    target = scalar_to_support(target_scalar, support_size)
    return -(target * torch.log_softmax(logits, dim=-1)).sum(dim=-1)

def ce_policy(logits, target_probs):
    return -(target_probs * torch.log_softmax(logits, dim=-1)).sum(dim=-1)

def muzero_loss(agent, batch, cfg):
    obs = torch.tensor(np.stack([sample[0] for sample in batch]), dtype=torch.float32)
    actions = torch.tensor(np.stack([sample[1] for sample in batch]), dtype=torch.long)
    tgt_v = torch.tensor(np.stack([sample[2] for sample in batch]), dtype=torch.float32)
    tgt_r = torch.tensor(np.stack([sample[3] for sample in batch]), dtype=torch.float32)
    tgt_p = torch.tensor(np.stack([sample[4] for sample in batch]), dtype=torch.float32)
    tgt_mask = torch.tensor(np.stack([sample[5] for sample in batch]), dtype=torch.float32)

    num_steps = actions.shape[1] + 1                       # K + 1 pas dépliés
    hidden, policy_logits, value_logits = agent.initial_inference(obs)
    value_loss = ce_support(value_logits, tgt_v[:, 0], agent.support_size)
    policy_loss = ce_policy(policy_logits, tgt_p[:, 0]) * tgt_mask[:, 0]
    reward_loss = torch.zeros(obs.shape[0])

    for k in range(actions.shape[1]):
        hidden, reward_logits, policy_logits, value_logits = agent.recurrent_inference(
            hidden, actions[:, k]
        )
        hidden.register_hook(lambda grad: grad * 0.5)      # 1/2 sur le gradient de la dynamique (papier)
        mask_k = tgt_mask[:, k + 1]
        value_loss = value_loss + ce_support(value_logits, tgt_v[:, k + 1], agent.support_size) * mask_k
        policy_loss = policy_loss + ce_policy(policy_logits, tgt_p[:, k + 1]) * mask_k
        reward_loss = reward_loss + ce_support(reward_logits, tgt_r[:, k + 1], agent.support_size) * mask_k

    # 1/K : on moyenne sur les pas dépliés pour que l'échelle soit indépendante de K
    value_loss = value_loss / num_steps
    policy_loss = policy_loss / num_steps
    reward_loss = reward_loss / num_steps

    total = cfg["value_loss_weight"]*value_loss + cfg["policy_loss_weight"]*policy_loss + reward_loss
    return total.mean(), {
        "value_loss": float(value_loss.mean().detach()),
        "policy_loss": float(policy_loss.mean().detach()),
        "reward_loss": float(reward_loss.mean().detach()),
    }

dummy_batch = rb.sample_batch(1, num_unroll_steps=2, td_steps=2, discount=0.9, num_actions=2)
tiny_agent = MuZeroAgent(_repr, _dyn, _pred, support_size=10, num_actions=2)
dummy_cfg = {"value_loss_weight": 0.25, "policy_loss_weight": 2.0}
loss, logs = tiny_agent.loss(dummy_batch, dummy_cfg)
for module in (_repr, _dyn, _pred):
    module.zero_grad()
loss.backward()
grad_norm = sum(float(p.grad.abs().sum()) for module in (_repr, _dyn, _pred) for p in module.parameters() if p.grad is not None)
assert math.isfinite(grad_norm) and grad_norm > 0.0
print("loss OK | grad_norm =", round(grad_norm, 3), "| logs =", logs)


**Lecture.** Le mini-test gradient compte plus qu'il n'en a l'air. Dans MuZero, les pertes
concernent des états latents déroules par `g`, pas seulement l'encodage initial. Vérifier qu'un
gradient non nul remonte bien jusqu'aux paramètres est une façon simple de s'assurer que le système
apprend quelque chose de bout en bout.

In [ ]:
train_cfg = {
    "num_simulations": 20,
    "discount": 0.997,
    "pb_c_base": 19652.0,
    "pb_c_init": 1.25,
    "dirichlet_alpha": 0.25,
    "exploration_fraction": 0.25,
    "two_player": False,
    "support_size": 25,
    "num_unroll_steps": 5,
    "td_steps": 10,
    "value_loss_weight": 0.25,
    "policy_loss_weight": 2.0,
    "batch_size": 64,
}

env_train = gym.make("CartPole-v1")
obs_dim = env_train.observation_space.shape[0]
num_actions = env_train.action_space.n

h = Representation(obs_dim, 32, 8)
g = Dynamics(8, num_actions, 32, train_cfg["support_size"])
f = Prediction(8, 32, num_actions, train_cfg["support_size"])
agent = MuZeroAgent(h, g, f, support_size=train_cfg["support_size"], num_actions=num_actions)
optimizer = torch.optim.Adam(agent.parameters(), lr=0.005, weight_decay=1e-4)
replay = ReplayBuffer(capacity=1000, seed=0)

_action_rng = np.random.default_rng(0)

def sample_action_from_visits(visits, temperature=1.0):
    if temperature <= 1e-8:
        return int(np.argmax(visits))
    scaled = np.power(visits + 1e-8, 1.0 / temperature)
    scaled = scaled / scaled.sum()
    return int(_action_rng.choice(len(scaled), p=scaled))   # RNG persistant (reproductible)

def play_game(agent, cfg, env, temperature=1.0, max_steps=200, seed=None):
    game = GameHistory()
    obs, _ = env.reset(seed=seed)                           # épisode seedé (reproductible)
    total_reward = 0.0
    for _ in range(max_steps):
        obs_tensor = torch.tensor(obs, dtype=torch.float32)
        root, search_value, priors, visits, _ = run_mcts(obs_tensor, agent, cfg, to_play=1, add_root_noise=True)
        action = sample_action_from_visits(visits, temperature=temperature)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        game.append(obs, action, reward, search_value, visits, to_play=1)   # F1 : valeur de recherche
        total_reward += float(reward)
        obs = next_obs
        if terminated or truncated:
            break
    return game, total_reward


In [ ]:
history = {
    "episode_return": [],
    "value_loss": [],
    "policy_loss": [],
    "reward_loss": [],
}

WARMUP_GAMES = 100
TRAIN_ITERATIONS = 800
UPDATES_PER_GAME = 8
REPORT_EVERY = 25

print(
    f"MuZero training | warmup={WARMUP_GAMES} parties | "
    f"train={TRAIN_ITERATIONS} parties | updates/partie={UPDATES_PER_GAME} | "
    f"simulations={train_cfg['num_simulations']} | batch={train_cfg['batch_size']}"
)

# ── Warmup : remplir le replay avec des parties exploratoires ───────────────
warmup_bar = tqdm(range(WARMUP_GAMES), desc="Warmup self-play", unit="partie")

for w in warmup_bar:
    warmup_game, warmup_return = play_game(
        agent,
        train_cfg,
        env_train,
        temperature=1.0,
        seed=2000 + w,
    )
    replay.save(warmup_game)
    history["episode_return"].append(float(warmup_return))

    warmup_bar.set_postfix(
        ret=f"{warmup_return:.1f}",
        mean10=f"{np.mean(history['episode_return'][-10:]):.1f}",
        replay=len(replay.games),
        positions=sum(len(g) for g in replay.games),
    )

print(
    f"Warmup terminé | replay={len(replay.games)} parties | "
    f"positions={sum(len(g) for g in replay.games)} | "
    f"retour moyen warmup={np.mean(history['episode_return'][-WARMUP_GAMES:]):.1f}"
)

# ── Training : self-play + updates réseau ───────────────────────────────────
train_bar = tqdm(range(TRAIN_ITERATIONS), desc="MuZero self-play + learn", unit="it")

for iteration in train_bar:
    if iteration < 0.50 * TRAIN_ITERATIONS:
        temp = 1.0
    elif iteration < 0.75 * TRAIN_ITERATIONS:
        temp = 0.5
    else:
        temp = 0.25

    game, total_reward = play_game(
        agent,
        train_cfg,
        env_train,
        temperature=temp,
        seed=iteration,
    )
    replay.save(game)
    history["episode_return"].append(float(total_reward))

    update_logs = {"value_loss": [], "policy_loss": [], "reward_loss": []}

    for _ in range(UPDATES_PER_GAME):
        batch = replay.sample_batch(
            batch_size=train_cfg["batch_size"],
            num_unroll_steps=train_cfg["num_unroll_steps"],
            td_steps=train_cfg["td_steps"],
            discount=train_cfg["discount"],
            num_actions=num_actions,
        )

        loss, logs = agent.loss(batch, train_cfg)

        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(agent.parameters(), 5.0)
        optimizer.step()

        for key in ("value_loss", "policy_loss", "reward_loss"):
            value = float(logs[key])
            history[key].append(value)
            update_logs[key].append(value)

    ret20 = float(np.mean(history["episode_return"][-20:]))
    ret100 = float(np.mean(history["episode_return"][-100:]))
    v_loss = float(np.mean(update_logs["value_loss"]))
    p_loss = float(np.mean(update_logs["policy_loss"]))
    r_loss = float(np.mean(update_logs["reward_loss"]))
    positions = sum(len(g) for g in replay.games)

    train_bar.set_postfix(
        ret=f"{total_reward:.1f}",
        mean20=f"{ret20:.1f}",
        mean100=f"{ret100:.1f}",
        temp=f"{temp:.2f}",
        v=f"{v_loss:.3f}",
        p=f"{p_loss:.3f}",
        r=f"{r_loss:.3f}",
        replay=len(replay.games),
        pos=positions,
    )

    if (iteration + 1) % REPORT_EVERY == 0:
        tqdm.write(
            f"[it {iteration + 1:03d}/{TRAIN_ITERATIONS}] "
            f"ret={total_reward:6.1f} | mean20={ret20:6.1f} | mean100={ret100:6.1f} | "
            f"temp={temp:.2f} | value={v_loss:.3f} policy={p_loss:.3f} reward={r_loss:.3f} | "
            f"replay={len(replay.games)} games / {positions} positions"
        )

env_train.close()

print(
    "Training terminé | "
    f"episodes={len(history['episode_return'])} | "
    f"last5={float(np.mean(history['episode_return'][-5:])):.2f} | "
    f"last20={float(np.mean(history['episode_return'][-20:])):.2f} | "
    f"last100={float(np.mean(history['episode_return'][-100:])):.2f}"
)

**Lecture.** Ce diagnostic sépare deux causes possibles d'échec. Si le rendement imaginé et le
rendement réel sont tous les deux faibles, le planner ne trouve simplement pas de bon plan. Si le
rendement imaginé est élevé mais le rendement réel chute, le planner exploite une erreur du modèle :
c'est le biais de modèle classique de PETS. Dans ce second cas, augmenter seulement CEM ne suffit pas ;
il faut améliorer le modèle, réduire l'horizon, augmenter les particules, ou collecter davantage de
données dans les régions visitées par le planner.

In [ ]:
def smooth(values, k=5):
    arr = np.asarray(values, dtype=np.float32)
    if len(arr) < k:
        return arr
    kernel = np.ones(k, dtype=np.float32) / k
    return np.convolve(arr, kernel, mode="valid")

fig, ax = plt.subplots(1, 4, figsize=(15, 3.3))
ax[0].plot(smooth(history["episode_return"]))
ax[0].axhline(22.0, color="black", linestyle=":", label="ordre de grandeur aleatoire")
ax[0].set_title("retour episode")
ax[0].legend()

ax[1].plot(smooth(history["value_loss"]))
ax[1].set_title("value loss")

ax[2].plot(smooth(history["policy_loss"]))
ax[2].set_title("policy loss")

ax[3].plot(smooth(history["reward_loss"]))
ax[3].set_title("reward loss")

for axis in ax:
    axis.set_xlabel("iteration")
fig.suptitle("CartPole : signal d'apprentissage a micro-echelle")
fig.tight_layout()
plt.show()

**Lecture.** Il faut lire ces courbes avec modestie. On ne cherche pas ici a "résoudre CartPole de
maniere spectaculaire". On cherche à vérifier que le pipeline complet est vivant : le replay se
remplit, les pertes restent finies, les retours bougent dans le bon sens, et la recherche nourrit bien
l'apprentissage de la politique. Pour un notebook from scratch, c'est déjà le bon signal.

Une bonne façon de lire ce mini-entraînement est de séparer **validation mécanique** et
**validation de performance**. La validation mécanique répond a des questions binaires :

- le replay collecte-t-il bien des épisodes complets ?
- les cibles n-step sont-elles alignees ?
- les pertes restent-elles finies ?
- les visites MCTS deviennent-elles une cible exploitable pour la politique ?

La validation de performance, elle, demanderait bien plus de budget et de reglages. Dans un gros
système MuZero, le niveau atteint depend fortement du nombre de simulations, de la profondeur des
deroulages, de la taille du replay, de la frequence des mises a jour et de la stabilité du réseau.
Ici, nous ne pretendons pas évaluer ce plafond. Nous montrons qu'un petit système cohérent peut déjà
apprendre quelque chose de non trivial en gardant chaque brique lisible.

In [ ]:
eval_env = gym.make("CartPole-v1")

obs_eval, _ = eval_env.reset(seed=11)
obs_eval_tensor = torch.tensor(obs_eval, dtype=torch.float32)

_, root_value_after, priors_after, visits_after, q_after = run_mcts(
    obs_eval_tensor,
    agent,
    train_cfg,
    add_root_noise=False,
)

returns = []
for seed in range(5):
    obs_rollout, _ = eval_env.reset(seed=100 + seed)
    total = 0.0

    for _ in range(500):
        _, _, _, visit_dist, _ = run_mcts(
            torch.tensor(obs_rollout, dtype=torch.float32),
            agent,
            train_cfg,
            add_root_noise=False,
        )
        action = int(np.argmax(visit_dist))

        obs_rollout, reward, terminated, truncated, _ = eval_env.step(action)
        total += float(reward)

        if terminated or truncated:
            break

    returns.append(total)

eval_env.close()

x_eval = np.arange(num_actions)

plt.figure(figsize=(6.4, 3.6))
plt.bar(x_eval - 0.2, priors_after, width=0.4, label="prior réseau")
plt.bar(x_eval + 0.2, visits_after, width=0.4, label="visites MCTS")
plt.xticks(x_eval, ["gauche", "droite"])
plt.ylabel("masse")
plt.title("Après apprentissage : prior et visites se parlent mieux")
plt.legend()
plt.show()

print(
    "eval returns:", returns,
    "| mean =", round(float(np.mean(returns)), 2),
    "| root value =", round(root_value_after, 3),
    "| q =", np.round(q_after, 3),
)

**Lecture.** On espere voir ici deux choses à la fois. D'abord, le prior réseau devient moins arbitraire
qu'au début : il imite davantage la structure des visites. Ensuite, le MCTS garde quand même son rôle
d'affinage local. Si prior et visites etaient toujours identiques, la recherche n'apporterait plus
grand-chose ; s'ils etaient totalement decorreles, le réseau n'apprendrait pas vraiment des parties.

## IV.4 — Diagnostics qui *prouvent* l'apprentissage

Une courbe de retour ne dit pas *d'où* vient le signal. On ajoute donc les diagnostics qui comptent vraiment : **le réseau seul fait-il aussi bien que le MCTS ?** (si oui, la recherche n'apporte rien), **la recherche aide-t-elle davantage avec plus de simulations ?**, la **valeur prédite** est-elle calibrée sur la cible n-step, et le **prior** imite-t-il les visites (KL).

In [ ]:
# Outil d'éval : politique greedy via la RECHERCHE (argmax visites) ou via le RÉSEAU SEUL (argmax prior)
@torch.no_grad()
def evaluate_policy(agent, cfg, n_episodes=5, num_sims=None, use_search=True, seed0=300, max_steps=500):
    eval_cfg = dict(cfg)
    if num_sims is not None:
        eval_cfg["num_simulations"] = num_sims
    env = gym.make("CartPole-v1")
    returns = []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + i)
        total = 0.0
        for _ in range(max_steps):
            if use_search:
                _, _, _, visits, _ = run_mcts(torch.tensor(obs, dtype=torch.float32), agent, eval_cfg, add_root_noise=False)
                action = int(np.argmax(visits))
            else:
                _, policy_logits, _ = agent.initial_inference(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))
                action = int(torch.argmax(policy_logits[0]))
            obs, reward, terminated, truncated, _ = env.step(action)
            total += float(reward)
            if terminated or truncated:
                break
        returns.append(total)
    env.close()
    return float(np.mean(returns)), float(np.std(returns))

print("evaluate_policy défini")

In [ ]:
# Diagnostic clé : la RECHERCHE apporte-t-elle quelque chose au-dessus du réseau seul ?
net_mean, net_std = evaluate_policy(agent, train_cfg, n_episodes=5, use_search=False)
mcts_mean, mcts_std = evaluate_policy(agent, train_cfg, n_episodes=5, use_search=True)

sim_budgets = [1, 5, 20, 50]
sweep = [evaluate_policy(agent, train_cfg, n_episodes=5, use_search=True, num_sims=s)[0] for s in sim_budgets]

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(["réseau seul\n(argmax prior)", "MCTS\n(argmax visites)"], [net_mean, mcts_mean],
          yerr=[net_std, mcts_std], capsize=6, color=["#94a3b8", "#2563eb"])
ax[0].set_ylabel("retour moyen (5 épisodes)")
ax[0].set_title("Réseau seul vs MCTS")
ax[1].plot(sim_budgets, sweep, marker="o", color="#2563eb")
ax[1].set_xlabel("nombre de simulations MCTS")
ax[1].set_ylabel("retour moyen")
ax[1].set_title("La recherche aide-t-elle plus avec plus de budget ?")
fig.tight_layout()
plt.show()
print(f"réseau seul = {net_mean:.1f} ± {net_std:.1f} | MCTS = {mcts_mean:.1f} ± {mcts_std:.1f}")
print("balayage simulations:", dict(zip(sim_budgets, [round(s, 1) for s in sweep])))

**Lecture.** Si la barre **MCTS** dépasse nettement **réseau seul**, la recherche apporte un vrai gain de décision (le réseau, encore imparfait, est corrigé par le calcul). Si les deux sont proches, soit le réseau a déjà internalisé la politique, soit la recherche est trop courte pour aider. Et la courbe du balayage dit si **augmenter les simulations** continue de payer — la signature même d'une planification utile.

In [ ]:
# Calibration : valeur prédite vs cible n-step, erreur de récompense par profondeur, KL(visites ‖ prior)
batch = replay.sample_batch(256, train_cfg["num_unroll_steps"], train_cfg["td_steps"], train_cfg["discount"], num_actions)
obs_b = torch.tensor(np.stack([b[0] for b in batch]), dtype=torch.float32)
act_b = torch.tensor(np.stack([b[1] for b in batch]), dtype=torch.long)
tgt_v = torch.tensor(np.stack([b[2] for b in batch]), dtype=torch.float32)
tgt_r = torch.tensor(np.stack([b[3] for b in batch]), dtype=torch.float32)
tgt_p = torch.tensor(np.stack([b[4] for b in batch]), dtype=torch.float32)
with torch.no_grad():
    hidden, policy_logits, value_logits = agent.initial_inference(obs_b)
    pred_v0 = support_to_scalar(value_logits, agent.support_size)
    prior0 = torch.softmax(policy_logits, dim=-1)
    rew_err, hk = [], hidden
    for k in range(act_b.shape[1]):
        hk, reward_logits, _, _ = agent.recurrent_inference(hk, act_b[:, k])
        pred_r = support_to_scalar(reward_logits, agent.support_size)
        rew_err.append(float((pred_r - tgt_r[:, k + 1]).abs().mean()))
eps = 1e-8
kl = float((tgt_p[:, 0] * (torch.log(tgt_p[:, 0] + eps) - torch.log(prior0 + eps))).sum(-1).mean())
agree = float((tgt_p[:, 0].argmax(-1) == prior0.argmax(-1)).float().mean())

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].scatter(tgt_v[:, 0].numpy(), pred_v0.numpy(), s=8, alpha=0.4)
lims = [float(min(tgt_v[:, 0].min(), pred_v0.min())), float(max(tgt_v[:, 0].max(), pred_v0.max()))]
ax[0].plot(lims, lims, "k--", lw=1)
ax[0].set_xlabel("cible n-step z_t"); ax[0].set_ylabel("valeur prédite"); ax[0].set_title("Calibration de la valeur")
ax[1].bar(range(1, len(rew_err) + 1), rew_err, color="#10b981")
ax[1].set_xlabel("profondeur de déroulage k"); ax[1].set_ylabel("|erreur récompense|"); ax[1].set_title("Erreur de récompense vs profondeur")
fig.tight_layout()
plt.show()
print(f"KL(visites ‖ prior) au pas 0 = {kl:.3f} | accord argmax(visites)==argmax(prior) = {agree:.2f}")

**Lecture.** Le **nuage valeur** doit se serrer autour de la diagonale : sinon la tête de valeur est mal calibrée (souvent par saturation du support — d'où `support_size=25`). L'**erreur de récompense** doit rester faible et ne pas exploser avec la profondeur `k` (sinon la dynamique apprise dérive vite). Enfin, **KL(visites ‖ prior)** qui baisse et l'**accord d'argmax** qui monte sont la vraie preuve que la tête de politique imite la recherche — bien plus parlant que la valeur brute de la policy-loss.

In [ ]:
# Démonstration CartPole : replay vidéo greedy via MCTS, sans fenêtre locale.
def evaluate_and_display_agent(label="MuZero CartPole", seed=999, max_steps=500, video_root="videos/14_muzero_cartpole"):
    safe_label = label.lower().replace(" ", "_").replace("-", "_")
    video_dir = Path(video_root) / safe_label
    video_dir.mkdir(parents=True, exist_ok=True)

    demo_env = gym.make("CartPole-v1", render_mode="rgb_array")
    demo_env = RecordVideo(
        demo_env,
        video_folder=str(video_dir),
        episode_trigger=lambda episode_id: episode_id == 0,
        name_prefix=f"{safe_label}_mcts",
    )

    obs, _ = demo_env.reset(seed=seed)
    total_reward = 0.0
    steps = 0

    try:
        for _ in range(max_steps):
            _, _, _, visits, _ = run_mcts(
                torch.tensor(obs, dtype=torch.float32),
                agent,
                train_cfg,
                add_root_noise=False,
            )
            action = int(np.argmax(visits))
            obs, reward, terminated, truncated, _ = demo_env.step(action)
            total_reward += float(reward)
            steps += 1
            if terminated or truncated:
                break
    finally:
        demo_env.close()

    print(f"Démo {label} | retour={total_reward:.0f} | durée={steps} pas")

    videos = sorted(video_dir.glob("*.mp4"), key=lambda path: path.stat().st_mtime)
    if videos:
        print(f"Replay vidéo {label} :", videos[-1])
        display(Video(str(videos[-1]), embed=True, width=480))
    else:
        print(f"Aucune vidéo générée pour {label}.")

    return total_reward


muzero_cartpole_demo = evaluate_and_display_agent()


**Pseudo-code de la boucle d'entraînement**

$$
\boxed{
\begin{aligned}
&\textbf{répéter :} \\
&\quad \text{jouer une partie avec MCTS} \to \text{stocker } (o_t,\, a_t,\, r_{t+1},\, \nu_t,\, \pi_t),\ \text{ajouter au replay} \\
&\quad \text{échantillonner un batch de positions} \\
&\quad s^0 = h(o);\quad p^0,\, v^0 = f(s^0) \\
&\quad \textbf{pour } k = 1 \dots K : \ s^k,\, r^k = g(s^{k-1},\, a_{k-1});\ \ p^k,\, v^k = f(s^k) \\
&\quad \mathcal{L} = \tfrac{1}{K+1}\textstyle\sum_k \big[\ell^\pi + \ell^v + \ell^r\big];\quad \text{rétroprop (gradient de dynamique} \times \tfrac{1}{2}) \\
\end{aligned}
}
$$

C'est cette alternance **recherche -> données -> apprentissage -> meilleure recherche** qui fait
l'identité de MuZero.

---
# Partie V — Connect-4 : MuZero en mode adversarial

CartPole nous a permis de construire tout le pipeline MuZero dans un problème **mono-joueur** :
une seule perspective, deux actions toujours légales et une récompense intermédiaire à chaque pas.

Il manque cependant une dimension historique importante de MuZero : son utilisation dans les jeux de
plateau adversariaux. Pour introduire cette difficulté sans passer directement aux échecs ou au Go,
nous utilisons **Connect-4**, le Puissance 4, fourni par
[PettingZoo `connect_four_v3`](https://pettingzoo.farama.org/environments/classic/connect_four/).

Connect-4 reste assez simple pour visualiser chaque décision, mais il ajoute exactement les mécanismes
qui nous manquent :

- deux joueurs dont les intérêts sont opposés ;
- une perspective qui change à chaque tour ;
- certaines actions qui deviennent illégales ;
- une récompense essentiellement terminale ;
- des décisions tactiques qui bénéficient réellement d'une recherche en arbre.

## V.1 — Règles de l'environnement

Le jeu utilise une grille de **6 lignes et 7 colonnes**. Les deux joueurs jouent à tour de rôle. Une
action consiste à choisir une colonne ; le jeton tombe alors jusqu'à la case libre la plus basse de
cette colonne.

```text
colonnes :   0   1   2   3   4   5   6
           ┌───┬───┬───┬───┬───┬───┬───┐
ligne 0    │   │   │   │   │   │   │   │
           ├───┼───┼───┼───┼───┼───┼───┤
ligne 1    │   │   │   │   │   │   │   │
           ├───┼───┼───┼───┼───┼───┼───┤
ligne 2    │   │   │   │   │   │   │   │
           ├───┼───┼───┼───┼───┼───┼───┤
ligne 3    │   │   │   │   │   │   │   │
           ├───┼───┼───┼───┼───┼───┼───┤
ligne 4    │   │   │   │   │   │   │   │
           ├───┼───┼───┼───┼───┼───┼───┤
ligne 5    │ X │ O │ X │ X │ O │   │   │
           └───┴───┴───┴───┴───┴───┴───┘
```

Le premier joueur qui aligne quatre jetons gagne. L'alignement peut être :

- horizontal ;
- vertical ;
- diagonal montant ;
- diagonal descendant.

La partie se termine également par un match nul lorsque les 42 cases sont occupées sans alignement
gagnant.

## V.2 — Actions et masque de légalité

L'espace d'actions est discret :

$$
\mathcal A=\{0,1,2,3,4,5,6\}.
$$

L'action correspond directement à l'indice de la colonne. Contrairement à CartPole, toutes les
actions ne restent pas disponibles pendant tout l'épisode : lorsqu'une colonne contient déjà six
jetons, elle devient pleine et il est interdit d'y jouer.

PettingZoo fournit donc un vecteur `action_mask` de longueur 7 :

```text
action_mask = [1, 1, 0, 1, 1, 0, 1]
```

Ici, les colonnes `2` et `5` sont pleines. Les autres colonnes restent jouables.

| Valeur du masque | Signification |
|---|---|
| `1` | l'action est légale pour le joueur courant |
| `0` | la colonne est pleine ou ce n'est pas au tour de cet agent |

Jouer une action illégale termine la partie et pénalise le joueur fautif. Le masque n'est donc pas une
simple aide numérique : il décrit une contrainte réelle du jeu.

À la racine du MCTS, MuZero multiplie les priorités du réseau par ce masque, puis renormalise les
actions restantes. Une colonne pleine reçoit ainsi exactement zéro priorité et zéro visite.

Dans l'arbre latent, la situation est plus subtile : MuZero n'a plus accès au véritable plateau ni au
simulateur PettingZoo. Conformément au papier, les actions illégales ne sont masquées qu'à la racine ;
le modèle apprend progressivement à ne pas proposer les coups qui n'apparaissent jamais dans les
trajectoires légales.

## V.3 — Observation : deux plans vus par le joueur courant

L'observation PettingZoo est un dictionnaire :

```python
{
    "observation": tableau de forme (6, 7, 2),
    "action_mask": tableau de forme (7,)
}
```

Le plateau contient deux plans binaires :

- `observation[:, :, 0]` indique les jetons du **joueur observé** ;
- `observation[:, :, 1]` indique les jetons de son **adversaire**.

Chaque case vaut `0` ou `1`. Une case vide vaut zéro dans les deux plans.

Cette représentation est relative au joueur dont on demande l'observation. Lorsque le tour change,
le premier plan signifie toujours « mes jetons » et le second « les jetons adverses ». Le réseau peut
donc partager les mêmes paramètres pour les deux camps sans devoir apprendre séparément la couleur
rouge et la couleur jaune.

Pour notre MLP, nous aplatissons simplement le tableau :

$$
6\times7\times2=84
$$

valeurs données à la fonction de représentation $h$.

Cette représentation conserve toute la géométrie du plateau, mais l'aplatissement n'exploite pas
explicitement sa structure spatiale. Une version plus ambitieuse utiliserait un réseau convolutionnel
ou résiduel, comme dans AlphaZero et MuZero original.

## V.4 — Récompenses et terminaison

Connect-4 ne fournit pas de récompense positive à chaque coup :

| Situation | Récompense |
|---|---:|
| coup non terminal | $0$ |
| victoire | $+1$ pour le gagnant |
| défaite | $-1$ pour le perdant |
| match nul | $0$ pour les deux joueurs |

Nous utilisons $\gamma=1$, car la longueur de la partie ne doit pas modifier la valeur d'une victoire :
gagner en 12 ou 14 coups reste une victoire de valeur `+1`.

Cette récompense terminale rend l'apprentissage plus difficile que dans CartPole. Au début de la
partie, aucune récompense immédiate n'indique directement quel coup était bon. La valeur doit propager
l'issue finale vers les positions antérieures, et le MCTS doit découvrir les conséquences tactiques
des actions.

## V.5 — Une valeur dépend toujours du joueur

Dans CartPole, une valeur positive est favorable au seul agent. Dans Connect-4, une même position peut
être excellente pour un joueur et catastrophique pour l'autre.

Nous représentons donc le joueur au trait par :

```text
player_0 → +1
player_1 → -1
```

Chaque nœud de l'arbre conserve son champ `to_play`. Si une feuille vaut `+1` pour le joueur qui doit
y jouer, elle vaut `-1` pour son adversaire au niveau précédent. Le backup doit donc inverser la
perspective à chaque profondeur.

```mermaid
flowchart LR
    classDef default fill:none,stroke:#64748b,stroke-width:1.5px
    state0["position actuelle<br/>joueur A"]
    actionA["joueur A choisit une colonne"]
    state1["nouvelle position<br/>joueur B"]
    actionB["joueur B répond"]
    state2["position suivante<br/>joueur A"]

    state0 --> actionA --> state1 --> actionB --> state2
    state2 -.->|"backup : perspective inversée"| state1
    state1 -.->|"backup : perspective inversée"| state0

    classDef playerA fill:none,stroke:#2563eb,stroke-width:2px
    classDef playerB fill:none,stroke:#dc2626,stroke-width:2px

    class state0,state2 playerA
    class state1 playerB
```

Cette alternance doit rester cohérente dans trois endroits :

1. le backup du MCTS ;
2. le score PUCT lu depuis le parent ;
3. les cibles de valeur construites depuis le replay.

## V.6 — API PettingZoo AEC

PettingZoo utilise ici son API **AEC** (*Agent Environment Cycle*). L'environnement indique
explicitement quel agent doit agir :

```python
for agent in env.agent_iter():
    observation, reward, terminated, truncated, info = env.last()

    if terminated or truncated:
        action = None
    else:
        action = choisir_action(observation)

    env.step(action)
```

`env.last()` retourne toujours les informations du joueur courant. Après `env.step(action)`,
PettingZoo passe automatiquement au joueur suivant.

Lorsque la partie est terminée, les agents apparaissent encore une dernière fois dans l'itération
afin de recevoir leur récompense finale. On doit alors appeler `env.step(None)` plutôt que fournir une
nouvelle action.

## V.7 — Pourquoi Connect-4 est un bon second laboratoire

| CartPole | Connect-4 |
|---|---|
| un agent | deux adversaires |
| deux actions toujours légales | sept actions, certaines deviennent illégales |
| récompense `+1` à chaque pas | récompense terminale `-1 / 0 / +1` |
| $\gamma=0{,}997$ | $\gamma=1$ |
| dynamique physique continue | dynamique discrète et déterministe |
| valeur dans une perspective unique | valeur dont le signe dépend du joueur |
| recherche de stabilisation | recherche tactique adversariale |

Connect-4 ne sert donc pas seulement à ajouter un second environnement. Il vérifie que notre
implémentation MuZero sait réellement gérer la structure pour laquelle MCTS est historiquement le
plus naturel : **imaginer plusieurs réponses adverses avant de choisir le prochain coup réel**.

## V.8 — Mini-expérience MuZero sur Connect-4

On entraîne ici un **vrai** petit MuZero adversarial par self-play à deux joueurs (perspective `to_play`, masque d'actions légales, backup à signe alterné), puis on l'évalue **honnêtement**. À cette micro-échelle le niveau atteint reste **incertain** — l'important est que le pipeline deux-joueurs soit correct. On regarde donc trois choses :

- les **pertes** de self-play descendent (le modèle apprend la dynamique et la valeur) ;
- la **performance vs un joueur aléatoire** sur plusieurs parties (victoires / nuls / défaites) — pas un score sur une seule partie ;
- une **vérification structurelle** : à la racine, les colonnes **illégales** reçoivent exactement zéro visite, et la perspective change bien de signe selon le joueur.

MuZero ne masque les actions illégales **qu'à la racine** ; dans l'arbre, il apprend à ne pas proposer les actions jamais observées, et il continue **au-delà des terminaux** via des états absorbants ([Schrittwieser et al., 2020 — MuZero](https://arxiv.org/abs/1911.08265), annexes).

In [ ]:
# Protocole Connect-4 et démo textuelle headless.
N_MATCH = 500
MUZERO_PLAYER = "player_1"


In [ ]:
# Mini-test : Connect-4, masque d'actions et encodage plateau
def board_obs_to_vector(observation):
    return observation["observation"].astype(np.float32).reshape(-1)

def legal_actions_from_mask(observation):
    return [i for i, flag in enumerate(observation["action_mask"]) if int(flag) == 1]

if HAS_PETTINGZOO:
    c4 = connect_four_v3.env()
    c4.reset(seed=0)
    observation, reward, terminated, truncated, info = c4.last()
    legal = legal_actions_from_mask(observation)
    board = observation["observation"].sum(axis=-1)
    assert observation["observation"].shape == (6, 7, 2)
    assert len(legal) == 7

    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].imshow(board, cmap="Blues")
    ax[0].set_title("plateau initial")
    ax[0].set_xticks(range(7))
    ax[0].set_yticks(range(6))
    ax[1].bar(range(7), observation["action_mask"])
    ax[1].set_title("action mask")
    ax[1].set_xticks(range(7), [f"c{i}" for i in range(7)])
    fig.tight_layout()
    plt.show()
    print("agent courant:", c4.agent_selection, "| legal actions:", legal)
    c4.close()
else:
    print("Section Connect-4 sautee :", PETTINGZOO_ERROR)


**Lecture.** Le `action_mask` vaut mieux qu'un simple détail d'API. Il rappelle que dans un jeu
adversarial, le MCTS ne peut pas se contenter d'un espace d'actions "théorique". La recherche doit
respecter les contraintes réelles du plateau. C'est aussi pourquoi l'expansion a été écrite plus haut
pour accepter explicitement une liste d'actions légales.

In [ ]:
if HAS_PETTINGZOO:
    torch.manual_seed(0); np.random.seed(0)
    _c4_rng = np.random.default_rng(0)
    agent4 = MuZeroAgent(
        Representation(obs_dim=84, hidden_dim=64, latent_dim=32),
        Dynamics(latent_dim=32, num_actions=7, hidden_dim=64, support_size=10),
        Prediction(latent_dim=32, hidden_dim=64, num_actions=7, support_size=10),
        support_size=10,
        num_actions=7,
    )
    opt4 = torch.optim.Adam(agent4.parameters(), lr=0.005, weight_decay=1e-4)
    cfg4 = {"num_simulations": 50, "discount": 0.997, "pb_c_base": 19652.0, "pb_c_init": 1.25,
            "dirichlet_alpha": 0.3, "exploration_fraction": 0.25, "two_player": True, "support_size": 10,
            "num_unroll_steps": 4, "td_steps": 16, "value_loss_weight": 0.25, "policy_loss_weight": 2.0, "batch_size": 32}
    replay4 = ReplayBuffer(capacity=1000, seed=0)
    PLAYER_SIGN = {"player_0": 1, "player_1": -1}

    def play_game_c4(muzero_agent, cfg, seed=None, add_root_noise=True):
        env = connect_four_v3.env()
        env.reset(seed=seed)

        game = GameHistory()
        outcome = {}

        for agent_name in env.agent_iter():
            obs_a, reward, terminated, truncated, _ = env.last()
            outcome[agent_name] = reward

            if terminated or truncated:
                env.step(None)
                continue

            legal = legal_actions_from_mask(obs_a)
            vec = torch.tensor(board_obs_to_vector(obs_a), dtype=torch.float32)

            _, search_value, _, visits, _ = run_mcts(
                vec,
                muzero_agent,
                cfg,
                legal_actions=legal,
                to_play=1,
                add_root_noise=add_root_noise,
            )

            probs = np.array([visits[k] for k in legal], dtype=np.float64)
            if probs.sum() > 0:
                action = legal[int(_c4_rng.choice(len(legal), p=probs / probs.sum()))]
            else:
                action = int(_c4_rng.choice(legal))

            game.append(
                board_obs_to_vector(obs_a),
                action,
                0.0,
                search_value,
                visits,
                to_play=PLAYER_SIGN[agent_name],
            )
            env.step(action)

        env.close()

        # Le résultat (+1) est porté par le dernier coup, dans la perspective du joueur qui l'a joué.
        if len(game) > 0 and (max(outcome.values()) if outcome else 0) == 1:
            game.rewards[-1] = 1.0

        return game
    # ---- self-play + apprentissage (budget modeste ; à micro-échelle le niveau reste incertain) ----
    C4_GAMES, C4_UPDATES = 800, 10
    for i in range(8):
        replay4.save(play_game_c4(agent4, cfg4, seed=i))
    c4_hist = {"value_loss": [], "policy_loss": [], "reward_loss": []}
    for it in range(C4_GAMES):
        replay4.save(play_game_c4(agent4, cfg4, seed=1000 + it))
        for _ in range(C4_UPDATES):
            b = replay4.sample_batch(cfg4["batch_size"], cfg4["num_unroll_steps"], cfg4["td_steps"], cfg4["discount"], 7)
            loss, logs = agent4.loss(b, cfg4)
            opt4.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(agent4.parameters(), 5.0)
            opt4.step()
        for key in c4_hist:
            c4_hist[key].append(logs[key])

    # ---- éval honnête : win-rate vs random + vérif structurelle ----
    def play_match(seed):
        opp = np.random.default_rng(10_000 + seed)
        env = connect_four_v3.env()
        env.reset(seed=seed)

        mcts_player = "player_0" if seed % 2 == 0 else "player_1"
        result = 0

        for agent_name in env.agent_iter():
            obs_a, reward, terminated, truncated, _ = env.last()

            if terminated or truncated:
                if agent_name == mcts_player:
                    result = reward
                env.step(None)
                continue

            legal = legal_actions_from_mask(obs_a)

            if agent_name == mcts_player:
                vec = torch.tensor(board_obs_to_vector(obs_a), dtype=torch.float32)
                _, _, _, visits, _ = run_mcts(
                    vec,
                    agent4,
                    cfg4,
                    legal_actions=legal,
                    to_play=1,
                    add_root_noise=False,
                )
                action = legal[int(np.argmax([visits[a] for a in legal]))]
            else:
                action = int(opp.choice(legal))

            env.step(action)

        env.close()
        return result

    results = [play_match(s) for s in range(N_MATCH)]
    wins = sum(1 for r in results if r == 1)
    losses_c4 = sum(1 for r in results if r == -1)
    draws = N_MATCH - wins - losses_c4

    env_chk = connect_four_v3.env(); env_chk.reset(seed=123)
    for _ in range(6):
        oo, _, t, tr, _ = env_chk.last()
        if t or tr:
            break
        env_chk.step(0)
    oo, *_ = env_chk.last(); legal_chk = legal_actions_from_mask(oo)
    _, _, _, visits_chk, _ = run_mcts(torch.tensor(board_obs_to_vector(oo), dtype=torch.float32), agent4, cfg4, legal_actions=legal_chk, to_play=1, add_root_noise=False)
    env_chk.close()
    illegal = [i for i in range(7) if i not in legal_chk]
    structural_ok = all(visits_chk[i] == 0.0 for i in illegal)

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(smooth(c4_hist["value_loss"]), label="value")
    ax[0].plot(smooth(c4_hist["policy_loss"]), label="policy")
    ax[0].set_title("Connect-4 : pertes d'entraînement (self-play)"); ax[0].set_xlabel("itération"); ax[0].legend()
    ax[1].bar(["victoires", "nuls", "défaites"], [wins, draws, losses_c4], color=["#22c55e", "#94a3b8", "#ef4444"])
    ax[1].set_title(f"Connect-4 entraîné vs random ({N_MATCH} parties)")
    fig.tight_layout(); plt.show()
    print(f"vs random : {wins} victoires / {draws} nuls / {losses_c4} défaites sur {N_MATCH}")
    print("vérif structurelle (visites = 0 sur actions illégales) :", structural_ok, "| illégales:", illegal)
else:
    print("Connect-4 non exécuté : PettingZoo indisponible.")


In [ ]:
# Démonstration Connect-4 headless : MuZero entraîné contre un joueur aléatoire.
# PettingZoo classic ne s'intègre pas aussi proprement à `RecordVideo` que Gymnasium.
# On garde donc une démo textuelle sans fenêtre locale ; CartPole fournit le replay vidéo du notebook.

def play_connect4_text_demo(seed=20_260):
    if not HAS_PETTINGZOO:
        print("Démo Connect-4 indisponible : PettingZoo n'est pas installé.")
        return None
    if "agent4" not in globals() or "cfg4" not in globals():
        print("Démo Connect-4 indisponible : lance d'abord la cellule d'entraînement Connect-4.")
        return None

    rng = np.random.default_rng(seed)
    demo_env = connect_four_v3.env()
    demo_env.reset(seed=0)
    terminal_rewards = None
    move_number = 0

    try:
        for agent_name in demo_env.agent_iter():
            observation, reward, terminated, truncated, _ = demo_env.last()

            if terminated or truncated:
                if terminal_rewards is None:
                    terminal_rewards = dict(demo_env.rewards)
                demo_env.step(None)
                continue

            legal_actions = legal_actions_from_mask(observation)
            if agent_name == MUZERO_PLAYER:
                board_vector = torch.tensor(board_obs_to_vector(observation), dtype=torch.float32)
                _, root_value, priors, visits, q_values = run_mcts(
                    board_vector,
                    agent4,
                    cfg4,
                    legal_actions=legal_actions,
                    to_play=1,
                    add_root_noise=False,
                )
                action = max(legal_actions, key=lambda column: visits[column])
                print(
                    f"coup {move_number:02d} | MuZero ({agent_name}) -> colonne {action} "
                    f"| valeur racine={root_value:+.3f} | visites={np.round(visits, 2)}"
                )
            else:
                action = int(rng.choice(legal_actions))
                print(f"coup {move_number:02d} | random ({agent_name}) -> colonne {action}")

            demo_env.step(action)
            move_number += 1

        terminal_rewards = terminal_rewards or dict(demo_env.rewards)

    finally:
        demo_env.close()

    muzero_reward = float(terminal_rewards.get(MUZERO_PLAYER, 0.0))
    if muzero_reward > 0:
        outcome = "victoire de MuZero"
    elif muzero_reward < 0:
        outcome = "défaite de MuZero"
    else:
        outcome = "match nul"

    print(f"Partie terminée après {move_number} coups : {outcome}.")
    print(f"Récompenses finales : {terminal_rewards}")
    return terminal_rewards


connect4_demo_rewards = play_connect4_text_demo()


**Lecture.** Le modèle a été entraîné par self-play : on lit donc une **vraie** courbe de pertes et un bilan vs random sur plusieurs parties, pas une curiosité de seed. La **vérification structurelle** (visites nulles sur les colonnes pleines) confirme que le masque de légalité et l'inversion de perspective fonctionnent. Si le bilan vs random reste modeste, c'est attendu à ce budget — on assume : ce qui est démontré ici, c'est la **correction du pipeline deux-joueurs**, pas un niveau de jeu.

## Ce qu'il faut retenir, et les pièges classiques

### Ce qu'on a construit

Dans ce chapitre, on a reconstruit le noyau de MuZero en partant de la question la plus simple :
**comment choisir une action quand on ne connaît pas encore bien les conséquences ?**

La progression est maintenant complète :

$$
\text{UCB1}
\;\longrightarrow\;
\text{UCT}
\;\longrightarrow\;
\text{PUCT}
\;\longrightarrow\;
(h,g,f)
\;\longrightarrow\;
\text{MCTS}
\;\longrightarrow\;
\text{cibles MuZero}
$$

MuZero combine trois idées :

| Idée | Rôle |
|---|---|
| **Modèle latent** | apprendre seulement ce qui sert à décider, pas reconstruire l'observation |
| **Réseau prior + valeur** | guider la recherche avec une intuition apprise |
| **MCTS** | améliorer cette intuition par calcul au moment de choisir l'action |

Le point central est que le réseau ne remplace pas la recherche. Il lui donne un point de départ :
un prior de politique, une valeur, et une dynamique latente. La recherche transforme ensuite ce prior
en une distribution de visites plus informée, qui devient la vraie cible de politique.

### Les deux démos

Sur **CartPole**, le notebook montre le pipeline mono-joueur : observations vectorielles, actions
discrètes, récompenses courtes, valeur n-step bootstrappée, et MCTS qui améliore progressivement le
prior réseau.

Sur **Connect-4**, on ajoute ce que CartPole ne peut pas montrer : actions légales, `to_play`,
perspective du joueur courant, et backup à signe alterné. Le but n'est pas d'obtenir un agent fort en
quelques secondes, mais de vérifier que les mécanismes adversariaux de MuZero sont en place.

Ces résultats doivent donc être lus honnêtement : ils prouvent la mécanique, pas une performance de
niveau papier.

### Pièges classiques

Les erreurs les plus fréquentes sont presque toujours des erreurs d'alignement ou de perspective :

- confondre la **politique réseau** avec la **cible de politique** : la cible est la distribution de
  visites MCTS, pas le prior brut ;
- décaler les récompenses : l'action `a_t` produit `r_{t+1}`, et les cibles doivent respecter cet
  alignement ;
- utiliser la valeur réseau comme bootstrap direct alors que MuZero stocke la **valeur de recherche**
  à la racine ;
- appliquer le bruit de Dirichlet ailleurs qu'à la racine ;
- oublier la normalisation min-max des Q dans le score PUCT ;
- oublier que dans un jeu à somme nulle, la valeur change de signe quand la perspective change ;
- sur-vendre un entraînement minuscule : ici, l'objectif est pédagogique.

### Ce que notre version simplifie

Les grandes implémentations de MuZero ajoutent beaucoup de couches pratiques :

- davantage de simulations par décision ;
- un replay beaucoup plus large, souvent priorisé ;
- du parallélisme massif entre self-play, entraînement et évaluation ;
- `reanalyze`, où les anciennes positions sont réévaluées par un réseau plus récent ;
- des variantes comme **Sampled MuZero** pour grands espaces d'actions ;
- des améliorations de stabilité comme **EfficientZero** ou **Gumbel MuZero**.

Mais ces raffinements ne changent pas le squelette. Ils cherchent surtout à produire de meilleures
cibles, à mieux dépenser le calcul, ou à adapter la recherche à des environnements moins dociles que
CartPole et Connect-4.

### Ce que vous devriez pouvoir réexpliquer

À la fin du notebook, vous devriez pouvoir répondre clairement à ces questions :

1. Pourquoi UCB1 contient déjà le compromis exploitation/exploration ?
2. Comment UCT transforme un bandit en arbre de recherche ?
3. Pourquoi PUCT multiplie le bonus d'exploration par un prior réseau ?
4. Quel est le rôle exact de `h`, `g` et `f` ?
5. Pourquoi MuZero ne reconstruit pas l'observation ?
6. Comment une distribution de visites devient une cible de politique ?
7. Comment se construit une cible de valeur n-step ?
8. Pourquoi le backup change de signe en deux joueurs ?
9. Ce que CartPole montre, et ce que Connect-4 ajoute.

Si cette chaîne est claire, vous avez la charpente intellectuelle de MuZero. Le reste est une question
de budget, d'ingénierie et de robustesse.

---

# Sources / Pour aller plus loin

- **Schrittwieser et al. (2020), *Mastering Atari, Go, Chess and Shogi by Planning with a Learned Model*.**  
  [arXiv:1911.08265](https://arxiv.org/abs/1911.08265) — le papier MuZero : fonctions `h/g/f`, modèle latent sans reconstruction, PUCT, cibles reward/value/policy et normalisation min-max des Q.

- **Silver et al. (2017), *Mastering the game of Go without human knowledge*.**  
  [Nature](https://www.nature.com/articles/nature24270) — AlphaGo Zero : MCTS guidé par un réseau policy/value, bruit de Dirichlet à la racine et apprentissage par self-play.

- **Silver et al. (2018), *A general reinforcement learning algorithm that masters chess, shogi, and Go through self-play*.**  
  [arXiv:1712.01815](https://arxiv.org/abs/1712.01815) — AlphaZero : généralisation du schéma policy/value + MCTS à plusieurs jeux.

- **Auer, Cesa-Bianchi & Fischer (2002), *Finite-time Analysis of the Multiarmed Bandit Problem*.**  
  [PDF](https://homes.di.unimi.it/~cesabian/Pubblicazioni/ml-02.pdf) — source UCB1 : bonus d'incertitude et regret logarithmique.

- **Kocsis & Szepesvári (2006), *Bandit Based Monte-Carlo Planning*.**  
  [PDF](https://ggp.stanford.edu/readings/uct.pdf) — introduction d'UCT : appliquer UCB localement dans chaque nœud d'un arbre.

- **Rosin (2011), *Multi-armed Bandits with Episode Context*.**  
  [PDF](https://chrisrosin.com/isaim2010final.pdf) — lignée PUCB/PUCT : biaiser l'exploration UCB avec un prédicteur.

- **Hubert et al. (2021), *Learning and Planning in Complex Action Spaces*.**  
  [arXiv:2104.06303](https://arxiv.org/abs/2104.06303) — Sampled MuZero : adaptation de MuZero à des espaces d'actions larges ou continus.

- **Ye et al. (2021), *Mastering Atari Games with Limited Data*.**  
  [arXiv:2111.00210](https://arxiv.org/abs/2111.00210) — EfficientZero : améliore fortement l'efficacité-échantillon de MuZero avec self-supervision, reanalyse et meilleures cibles.

- **Danihelka et al. (2021), *Policy Improvement by Planning with Gumbel*.**  
  [arXiv:2111.07954](https://arxiv.org/abs/2111.07954) — Gumbel MuZero : variante de recherche qui améliore la sélection d'actions et la politique cible.

- **Werner Duvaud, `muzero-general`.**  
  [GitHub](https://github.com/werner-duvaud/muzero-general) — implémentation pédagogique très utile pour comparer les choix pratiques sur CartPole et Connect-4.

- **PettingZoo — Connect Four.**  
  [Documentation](https://pettingzoo.farama.org/environments/classic/connect_four/) — API multi-agent utilisée pour la mini-démo adversariale.

**Renvois dans cette série.** Le notebook [08](./08_sac_halfcheetah_walkthrough.ipynb) (SAC) donne le contraste avec le contrôle continu off-policy, le notebook [11](./11_pets_halfcheetah_walkthrough.ipynb) (PETS) montre la planification MPC avec un modèle dynamique explicite, le notebook [12](./12_mbpo_halfcheetah_walkthrough.ipynb) (MBPO) utilise un modèle pour entraîner une politique, et le notebook [13](./13_dreamer_walkthrough.ipynb) (Dreamer) sert de comparaison directe pour les world models latents avec reconstruction et imagination.